# BERTopic + Image Topic Scoring.

## Inputs (already prepared)
- We start with **4 CSVs**: `kremlin_en`, `kremlin_ru` (has English translations), `mid_en`, `mid_ru` (has English translations).
- Each CSV already includes scraped/extracted fields like:
  `id, url, date/year/month/day/time, title, title_english, full_text, full_text_english, full_text_word_count,
   location/lat/lon, speakers, image_captions, page_summary, declared_*`
- Images are linked via `stored_image_filepaths`
- **RU rule:** run BERTopic on **`full_text_english` only** (not Russian) so topics align with EN resolution.

---

## 1) Select dataset (run one corpus at a time)
Set:
- `DATASET_NAME ∈ {kremlin_en, kremlin_ru, mid_en, mid_ru}`
- `CSV_PATH`
- `ID_COL = "id"`
- `TEXT_COL`:
  - EN corpora → `full_text`
  - RU corpora → `full_text_english`
- Images come from `stored_image_filepaths` (list/delimited string).

---

## 2) Fit base BERTopic (text only)
Steps:
- Load CSV → light text cleanup.
- Chunk long docs if needed.
- Create embeddings (SentenceTransformer).
- Fit BERTopic → base topics (`base_k`).
Outputs in session:
- `base_model`, `text_embeddings`, `topics_base`, `probs_base`.

---

## 3) K-sweep (EN only)
Run only for:
- Kremlin EN
- MID EN
Logic:
- `K_SWEEP_LIST = 200, 190, …, 10` (descending).
- Keep only `k <= base_k` (since `reduce_topics()` only reduces).
For each K:
- compute coherence (c_npmi), diversity, compactness, separation, composite score.
Save:
- `k_sweep_metrics.csv` + scree plot.
Choose:
- `best_k_kremlin_en` (e.g., 89)
- `best_k_mid_en` (e.g., 32)

---

## 4) Reuse EN-chosen K for RU
No sweep for RU.
Set:
- Kremlin RU → `K = best_k_kremlin_en`
- MID RU → `K = best_k_mid_en`
Reason: RU uses English translations, so we force the same topic resolution.

---

## 5) Fit final BERTopic at chosen K (all 4 corpora)
For each dataset:
- Fit or reduce to selected `K`.
- Produce final `topics_final` and `probs_final`.
Exports:
1. `speech_topk_topics.csv` (wide: per speech top topic + prob)
2. `doc_topic_probs_long.csv` (long: `N_docs × K`)
3. `topics.csv` (topic id + keywords + counts)

 Image embeddings + image-topic scoring (CLIP)
For each dataset:
- Resolve images per speech via `stored_image_filepaths`.
- Embed images using CLIP (cache vectors).
- Build CLIP text vectors from topic keywords.
- Score each image vs each topic → softmax probabilities.
Exports:
1. `image_embeddings.parquet` (id, image_path, vec_path)
2. `image_topic_probs_long.csv` (long: `N_images × K`)

Build HTML topic browser (topics + images)
For each dataset:
- Use `topics.csv` + `speech_topk_topics.csv` (+ cached CLIP scores).
- Show top speeches per topic + top images per topic.
Export:
- `topics_all_in_one.html` (used for naming topics + assigning groups)

---

## 6) Add curated columns (final merge)
After topic labels + group names are finalized:
Text curated columns:
- `curated_topic_id`
- `curated_text_topic_label`
- `curated_text_topic_group`
- `curated_topic_probability`

Image curated columns:
- `curated_image_topic_ids`
- `curated_image_topic_labels`
- `curated_image_group_names`
- `curated_image_topic_probabilities`

Then produce the final corpus CSVs with curated columns.

---

## 7) Final deliverables
Final CSVs:
- `kremlin_english.csv`
- `kremlin_russian.csv`
- `mid_english.csv`
- `mid_russian.csv`

Supporting files:
- `doc_topic_probs_long.csv` (heavy)
- `image_topic_probs_long.csv` (heavy)
- `topics_all_in_one.html`
- EN only: `k_sweep_metrics.csv` + scree plot


## Environment setup (Python + required packages)

- we will install these required packages
  - BERTopic
  - gensim coherence (c_npmi)
  - UMAP/HDBSCAN
  - sentence-transformers (text + CLIP)
  - pyarrow (parquet)



# Install Packages and Import them

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1) Clean restart recommended (Runtime > Restart runtime). Then run:
%pip install -q --upgrade \
  pandas==2.2.2 \
  numpy>=2,<2.3 \
  scikit-learn>=1.5,<1.6 \
  umap-learn==0.5.6 \
  hdbscan==0.8.40 \
  bertopic==0.16.3 \
  sentence-transformers==3.0.1 \
  transformers==4.44.2 \
  pillow>=10.4 \
  tqdm>=4.67

In [ ]:
%%python
# Compatibility shim
import sys, os, textwrap
import numpy as np
if not hasattr(np, "infty"):
    setattr(np, "infty", np.inf)

# Try imports
mods = ["numba","llvmlite","hdbscan","umap","bertopic","transformers","sklearn","pandas"]
ok = True
for m in mods:
    try:
        __import__(m)
        print("imported", m)
    except Exception as e:
        print("FAILED:", m, "->", repr(e))
        ok = False
        break
print("ALL GOOD" if ok else "FIX NEEDED")


In [ ]:
!pip -q install "bertopic==0.16.2" "sentence-transformers>=3.0.1" "transformers>=4.44.0" \
                "umap-learn==0.5.6" "hdbscan>=0.8.40" "pandas>=2.2.2" "scikit-learn>=1.5.2" \
                "numpy>=2.0.2" "pyarrow>=16.1.0" "tqdm>=4.67.0" "Pillow>=10.3.0"

In [ ]:
!pip -q install "jedi>=0.19.1"

In [ ]:
!pip install -q --no-deps bertopic==0.16.4

In [ ]:
import os, sys, re, json, base64, logging, glob, hashlib
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm import tqdm

from PIL import Image, ImageFile, ImageOps
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from google.colab import drive

#Fit the Base BERT Topic Model to get all of the possible Topics, then we will apply K sweep on top of this to get optimal number of topics for the kremlin/mid english datasets.

In [ ]:

CSV_PATH   = "/content/drive/MyDrive/Russian Speech Dataset Project/CSV Files/mid_english.csv"
IMAGE_ROOT = "/content/drive/MyDrive/Russian Speech Dataset Project/Archive/New Files/Mid/English/Scraped Images"
OUTPUT_DIR = "/content/drive/MyDrive/English/csv files/Final CSV Files/bertopic_text_baseline_outputs_jan_rakesh"

# Optional: if URL not in CSV, build from ID
URL_TEMPLATE = ""  # e.g., "https://en.kremlin.ru/events/president/news/{id}"

# Create dirs
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "image_vecs"), exist_ok=True)

# MODEL CONFIG
TEXT_MODEL  = "sentence-transformers/all-mpnet-base-v2"
CLIP_MODEL  = "clip-ViT-B-32"

# Chunking (cover full text via windows under 512)
CHUNK_BODY          = 448
CHUNK_STRIDE        = 128
MAX_CHUNKS_PER_DOC  = 32

# Batching
BATCH_TEXT   = 32
BATCH_IMAGES = 96

# BERTopic
MIN_TOPIC_SIZE   = 12
TOP_N_TOPICS     = None         # export ALL non -1 topics
IMAGES_PER_TOPIC = 10
MAX_IMAGES_PER_SPEECH = 8       # for reranking cost control

# Vectorizer / c-TF-IDF
VECTORIZER = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3)
CTFIDF     = ClassTfidfTransformer(reduce_frequent_words=True)

# Repro
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Logging / device
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("bertopic_text_only")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
log.info(f"Device: {DEVICE}")

# Mount Drive (if not yet)
try:
    drive.mount('/content/drive')
except Exception:
    pass

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def fail_if_not_utf8(path: str):
    with open(path, "rb") as f:
        data = f.read()
    try:
        data.decode("utf-8")
    except UnicodeDecodeError:
        log.error(f"CSV is not UTF-8: {path}")
        sys.exit(1)

def read_csv_utf8(path: str) -> pd.DataFrame:
    fail_if_not_utf8(path)
    df = pd.read_csv(path, encoding="utf-8")

    df["ID"] = df["id"].astype(str)
    df["full_text"] = df["full_text"].astype(str)
    return df

def simple_text_clean(t: str) -> str:
    t = t.replace("\xa0", " ")
    t = re.sub(r"<[^>]+>", " ", t)     # strip HTML
    t = re.sub(r"\s+", " ", t).strip()
    return t

def list_images_for_id(speech_id) -> list[str]:
    folder = os.path.join(IMAGE_ROOT, str(speech_id).strip())
    paths = []
    if os.path.isdir(folder):
        for p in sorted(glob.glob(os.path.join(folder, "*"))):
            if os.path.splitext(p.lower())[1] in IMG_EXTS and os.path.exists(p):
                paths.append(p)
    return paths

def split_images_cell(cell: str) -> list[str]:
    paths = []
    for part in str(cell).split("||"):
        part = part.strip()
        if not part:
            continue
        cand = os.path.join(IMAGE_ROOT, part) if not os.path.isabs(part) else part
        if os.path.exists(cand):
            paths.append(cand)
    return paths

def prefer_images_for_row(row) -> list[str]:
    # Prefer an explicit "image_filenames" column if present; otherwise folder-per-ID
    if "image_filenames" in row and str(row["image_filenames"]).strip():
        paths = split_images_cell(row["image_filenames"])
        if paths:
            return paths
    return list_images_for_id(row["ID"])

def load_resize_image(path: str, size=(512, 512)):
    try:
        resample = getattr(Image, "LANCZOS", Image.BICUBIC)
        im = Image.open(path).convert("RGB")
        im = ImageOps.exif_transpose(im)
        im = im.resize(size, resample)
        return im
    except Exception as e:
        log.warning(f"Image load failed '{path}': {e}")
        return None

def encode_png_base64(im: Image.Image, size=(384, 384)) -> str:
    im = ImageOps.exif_transpose(im).convert("RGB")
    resample = getattr(Image, "LANCZOS", Image.BICUBIC)
    im = im.resize(size, resample).copy()
    buf = BytesIO()
    im.save(buf, format="PNG", optimize=False)   # PNG avoids Pillow JPEG fileno issue
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def l2norm(mat: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(mat, axis=1, keepdims=True)
    n[n == 0.0] = 1.0
    return mat / n

def build_chunk_texts(text: str, tokenizer: AutoTokenizer) -> list[str]:
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    if not ids:
        return []
    chunks = []
    for start in range(0, len(ids), CHUNK_STRIDE):
        window = ids[start:start + CHUNK_BODY]
        if not window:
            break
        ch = tokenizer.decode(window, skip_special_tokens=True).strip()
        if ch:
            chunks.append(ch)
        if len(chunks) >= MAX_CHUNKS_PER_DOC:
            break
    if not chunks:
        chunks = [text[:2000]]
    return chunks

def topic_prompt(keyword_string: str) -> str:
    return f"news photo of {keyword_string}"

def vec_path_for_image(image_path: str) -> str:
    # stable filename in OUTPUT_DIR/image_vecs/
    h = hashlib.md5(image_path.encode("utf-8")).hexdigest()
    base = f"{h}.npy"
    return os.path.join(OUTPUT_DIR, "image_vecs", base)

def main():
    # ---- Load CSV
    log.info("Reading CSV…")
    df = read_csv_utf8(CSV_PATH).copy()
    df["full_text"] = df["full_text"].map(simple_text_clean)

    # ---- Resolve images (but DO NOT drop rows without images)
    log.info("Align images…")
    all_paths = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Align images"):
        all_paths.append(prefer_images_for_row(row))
    df["all_image_paths"] = all_paths

    # ---- Text model and tokenizer
    log.info("Loading text model + tokenizer…")
    text_model = SentenceTransformer(TEXT_MODEL, device=DEVICE)
    tokenizer  = AutoTokenizer.from_pretrained(TEXT_MODEL, use_fast=True)

    # ---- Chunk + embed text (cache all to Drive)
    log.info("Chunking…")
    chunk_lists = []
    for t in tqdm(df["full_text"].tolist(), total=len(df), desc="Chunking"):
        chunk_lists.append(build_chunk_texts(t, tokenizer))

    log.info("Encoding text chunks…")
    dim = text_model.get_sentence_embedding_dimension()
    text_doc_vecs = np.zeros((len(df), dim), dtype=np.float32)

    work = []
    for i, chunks in enumerate(chunk_lists):
        for ch in chunks:
            work.append((i, ch))
    results_per_doc = [[] for _ in range(len(df))]

    with tqdm(total=len(work), desc="Encode text chunks") as bar:
        cursor = 0
        while cursor < len(work):
            end = min(cursor + BATCH_TEXT, len(work))
            owners = [work[j][0] for j in range(cursor, end)]
            texts  = [work[j][1] for j in range(cursor, end)]
            vecs = text_model.encode(
                texts, batch_size=len(texts), device=DEVICE,
                show_progress_bar=False, convert_to_numpy=True,
                normalize_embeddings=False
            )
            for o, v in zip(owners, vecs):
                results_per_doc[o].append(v.astype(np.float32))
            cursor = end
            bar.update(len(texts))

    for i in range(len(df)):
        if results_per_doc[i]:
            text_doc_vecs[i] = np.mean(results_per_doc[i], axis=0)
        else:
            v = text_model.encode([df.loc[i,"full_text"][:2000]], device=DEVICE,
                                  show_progress_bar=False, convert_to_numpy=True)
            text_doc_vecs[i] = v[0].astype(np.float32)

    # Save text embeddings + doc index (for future reuse keyed by ID)
    np.savez_compressed(os.path.join(OUTPUT_DIR, "text_embeddings.npz"), X=text_doc_vecs)
    pd.DataFrame({"row_idx": np.arange(len(df)), "ID": df["ID"]}).to_csv(
        os.path.join(OUTPUT_DIR, "doc_index.csv"), index=False
    )
    log.info("Saved text_embeddings.npz and doc_index.csv")

    # ---- Fit BERTopic (text-only)
    log.info("Fitting BERTopic (text-only)…")
    topic_model = BERTopic(
        embedding_model=None,
        vectorizer_model=VECTORIZER,
        ctfidf_model=CTFIDF,
        min_topic_size=MIN_TOPIC_SIZE,
        nr_topics=None,
        calculate_probabilities=True,
        low_memory=True,
        verbose=True
    )
    docs = df["full_text"].tolist()
    topics, probs = topic_model.fit_transform(docs, embeddings=text_doc_vecs)

    # ---- Topic info (ALL non -1 topics)
    info = topic_model.get_topic_info()
    info = info[info["Topic"] != -1].sort_values("Count", ascending=False).copy()

    def kw_for(tid: int) -> str:
        pairs = topic_model.get_topic(int(tid)) or []
        return ", ".join([w for (w, _) in pairs])

    info["Keywords"] = info["Topic"].map(kw_for)
    topics_df = info.rename(columns={"Topic":"topic_id","Name":"topic_name","Count":"count","Keywords":"keywords"})
    topics_df = topics_df[["topic_id","topic_name","keywords","count"]]
    topics_df.to_csv(os.path.join(OUTPUT_DIR,"topics.csv"), index=False)
    log.info("Saved topics.csv")

    # ---- Speech → top topic mapping (ALL rows, even if prob low)
    probs_np = np.array(probs) if probs is not None else None
    topic2kw = {int(r["topic_id"]): r["keywords"] for _, r in topics_df.iterrows()}

    speech_rows = []
    for i in range(len(df)):
        sid = df.loc[i, "ID"]
        assigned_tid = int(topics[i])
        assigned_prob = None
        if probs_np is not None and probs_np.ndim == 2 and i < probs_np.shape[0]:
            assigned_prob = float(np.max(probs_np[i]))
        url = None
        if "url" in df.columns and isinstance(df.loc[i, "url"], str) and df.loc[i, "url"].strip():
            url = df.loc[i,"url"]
        elif URL_TEMPLATE:
            url = URL_TEMPLATE.format(id=sid)
        speech_rows.append({
            "ID": sid,
            "top_topic": assigned_tid,
            "top_prob": assigned_prob,
            "topic_keywords": topic2kw.get(assigned_tid, ""),
            "url": url
        })
    speech_topk = pd.DataFrame(speech_rows)
    speech_topk.to_csv(os.path.join(OUTPUT_DIR, "speech_topk_topics.csv"), index=False)
    log.info("Saved speech_topk_topics.csv")

    # Representative docs HTML (preview)
    rows = []
    for tid in topics_df["topic_id"].astype(int).tolist():
        reps = topic_model.get_representative_docs(int(tid)) or []
        for doc in reps[:3]:
            prev = (doc[:600] + "…") if len(doc) > 600 else doc
            rows.append({"topic_id": int(tid), "preview": prev})
    rep_df = pd.DataFrame(rows)
    with open(os.path.join(OUTPUT_DIR, "representative_docs.html"), "w", encoding="utf-8") as f:
        f.write(rep_df.to_html(index=False, escape=True))
    log.info("Saved representative_docs.html")

    # Image embeddings (cache by ID+path to Drive)
    log.info("Embedding images with CLIP (cache & reuse)…")
    clip_model = SentenceTransformer(CLIP_MODEL, device=DEVICE)

    # We will create/append a manifest: (ID, image_path, vec_npy_path)
    manifest_path = os.path.join(OUTPUT_DIR, "image_embeddings.parquet")
    if os.path.exists(manifest_path):
        img_manifest = pd.read_parquet(manifest_path)
    else:
        img_manifest = pd.DataFrame(columns=["ID","image_path","vec_path"])

    existing = set((row["ID"], row["image_path"]) for _, row in img_manifest.iterrows())
    new_rows = []

    # Load representative first image per speech for BERTopic visual rep (optional later) and
    # also prepare all image paths for per-topic reranking
    rep_imgs = [paths[0] if len(paths)>0 else None for paths in df["all_image_paths"]]

    # Batch-embed ONLY missing images, save per-image vector to .npy file in OUTPUT_DIR/image_vecs/
    to_embed = []
    id_for = []
    for i in range(len(df)):
        sid = df.loc[i, "ID"]
        for p in df.loc[i,"all_image_paths"][:MAX_IMAGES_PER_SPEECH]:
            key = (sid, p)
            if key not in existing:
                to_embed.append(p)
                id_for.append(sid)

    # Split into chunks to not overflow memory
    for start in tqdm(range(0, len(to_embed), BATCH_IMAGES), desc="Image batches"):
        batch_paths = to_embed[start:start+BATCH_IMAGES]
        images = [load_resize_image(p) for p in batch_paths]
        keep = [(p, im) for p, im in zip(batch_paths, images) if im is not None]
        if not keep:
            continue
        batch_paths2, ims2 = zip(*keep)
        vecs = clip_model.encode(list(ims2), batch_size=min(BATCH_IMAGES, len(ims2)),
                                 device=DEVICE, show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
        for pth, vec in zip(batch_paths2, vecs):
            vecf = vec_path_for_image(pth)
            np.save(vecf, vec)
            sid = id_for[to_embed.index(pth)]  # safe since unique in 'to_embed'
            new_rows.append({"ID": sid, "image_path": pth, "vec_path": vecf})

    if new_rows:
        img_manifest = pd.concat([img_manifest, pd.DataFrame(new_rows)], ignore_index=True)
        img_manifest.drop_duplicates(subset=["ID","image_path"], inplace=True)
        img_manifest.to_parquet(manifest_path, index=False)
    log.info(f"Saved/updated image_embeddings.parquet with {len(img_manifest)} rows")

    # Build topic prompts (from keywords) and embed once
    kw_map = {int(r["topic_id"]): r["keywords"] for _, r in topics_df.iterrows()}
    prompt_vecs = []
    prompt_topic_ids = []
    prompts = []
    for tid, kws in kw_map.items():
        prompt = topic_prompt(kws)
        prompts.append(prompt)
        prompt_topic_ids.append(int(tid))
    if prompts:
        pvecs = clip_model.encode(prompts, device=DEVICE, show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
        pvecs = l2norm(pvecs)
        prompt_vecs = {tid: pvecs[i] for i, tid in enumerate(prompt_topic_ids)}
    else:
        prompt_vecs = {}

    # Rerank images per topic (cosine(prompt,image))
    def rerank_images_for_topic(tid: int, topk: int = IMAGES_PER_TOPIC, cap: int = 300):
        if tid not in prompt_vecs:
            return []
        pvec = prompt_vecs[tid]
        cand = []

        # collect members (speech indices) for tid by max-prob ranking
        probs_np_local = np.array(probs) if probs is not None else None
        members = []
        for i, t in enumerate(topics):
            if t == tid:
                pr = float(np.max(probs_np_local[i])) if probs_np_local is not None and probs_np_local.ndim == 2 else 1.0
                members.append((pr, i))
        members.sort(key=lambda x: -x[0])

        # walk members and their images
        for pr, row_idx in members:
            sid = df.loc[row_idx, "ID"]
            row_imgs = img_manifest[img_manifest["ID"] == sid]
            if row_imgs.empty:
                continue
            for _, r in row_imgs.head(MAX_IMAGES_PER_SPEECH).iterrows():
                vec = np.load(r["vec_path"]).astype(np.float32)
                vec = vec / (np.linalg.norm(vec) + 1e-12)
                score = float(np.dot(pvec, vec))
                link = None
                if "url" in df.columns and isinstance(df.loc[row_idx, "url"], str) and df.loc[row_idx, "url"].strip():
                    link = df.loc[row_idx, "url"]
                elif URL_TEMPLATE:
                    link = URL_TEMPLATE.format(id=sid)
                cand.append((score, sid, r["image_path"], link))
                if len(cand) >= cap:
                    break
            if len(cand) >= cap:
                break
        cand.sort(key=lambda x: -x[0])
        return cand[:topk]

    # Build big HTML with 10 images/topic + top speeches + reps
    def make_tile(path, sid, link, thumb_w=200):
        im = load_resize_image(path, size=(384,384))
        if im is None: return ""
        b64 = encode_png_base64(im, size=(384, 384))
        link_html = f' — <a href="{link}" target="_blank">open</a>' if link else ""
        return (
            f'<figure style="width:{thumb_w}px;margin:6px;">'
            f'  <img loading="lazy" src="data:image/png;base64,{b64}" '
            f'       style="width:{thumb_w}px;border:1px solid #ddd;border-radius:6px;">'
            f'  <figcaption style="font:12px/1.35 system-ui;color:#444;margin-top:4px;">ID <b>{sid}</b>{link_html}</figcaption>'
            f'</figure>'
        )

    # Precompute topic → members (top speeches table)
    topic_members = {}
    probs_np = np.array(probs) if probs is not None else None
    for i, t in enumerate(topics):
        if t == -1:
            continue
        pr = float(np.max(probs_np[i])) if probs_np is not None and probs_np.ndim == 2 else 1.0
        topic_members.setdefault(int(t), []).append((pr, i))
    for t in topic_members:
        topic_members[t].sort(key=lambda x: -x[0])

    sections = []
    for _, r in topics_df.iterrows():
        tid   = int(r["topic_id"])
        name  = r["topic_name"]
        count = int(r["count"])
        keys  = r["keywords"]

        ranked = rerank_images_for_topic(tid, IMAGES_PER_TOPIC, cap=300)
        tiles  = [make_tile(p, sid, link) for (score, sid, p, link) in ranked]
        gallery = ''.join(tiles) if tiles else '<div style="color:#777;">No images available</div>'

        # Top speeches (first 6)
        rows_html = []
        for pr, idx in topic_members.get(tid, [])[:6]:
            sid = df.loc[idx, "ID"]
            if "url" in df.columns and isinstance(df.loc[idx, "url"], str) and df.loc[idx, "url"].strip():
                url_html = f'<a href="{df.loc[idx, "url"]}" target="_blank">{sid}</a>'
            elif URL_TEMPLATE:
                url_html = f'<a href="{URL_TEMPLATE.format(id=sid)}" target="_blank">{sid}</a>'
            else:
                url_html = f"{sid}"
            rows_html.append(
                f"<tr><td style='padding:6px 10px;border-bottom:1px solid #eee'>{url_html}</td>"
                f"<td style='padding:6px 10px;border-bottom:1px solid #eee;text-align:right'>{pr:.3f}</td></tr>"
            )
        top_table = (
            "<table style='border-collapse:collapse;font:13px system-ui;'>"
            "<thead><tr><th style='text-align:left;padding:6px 10px;border-bottom:1px solid #ddd'>Speech ID</th>"
            "<th style='text-align:right;padding:6px 10px;border-bottom:1px solid #ddd'>Assigned prob</th></tr></thead>"
            f"<tbody>{''.join(rows_html)}</tbody></table>"
        )

        # Representative text excerpts
        reps = topic_model.get_representative_docs(tid) or []
        rep_blocks = []
        for doc in reps[:3]:
            prev = (doc[:600] + "…") if len(doc) > 600 else doc
            rep_blocks.append(f"<div style='margin:8px 0;padding:10px;background:#fafafa;border:1px solid #eee;border-radius:6px;'>{prev}</div>")
        reps_html = ''.join(rep_blocks) if rep_blocks else "<div style='color:#777;'>No previews</div>"

        section_html = f"""
        <section id="topic-{tid}" style="margin:24px 0 36px 0;padding-top:24px;border-top:2px solid #f2f2f2;">
          <h2 style="margin:0 0 6px 0;">Topic {tid} — {name}</h2>
          <div style="color:#666;margin:0 0 12px 0;">Count: <b>{count}</b></div>
          <div style="color:#444;margin:0 0 14px 0;"><b>Keywords:</b> {keys}</div>
          <div style="display:flex;flex-wrap:wrap;gap:6px;">{gallery}</div>
          <details style="margin-top:14px;">
            <summary style="cursor:pointer;">Top speeches (first 6)</summary>
            <div style="margin-top:10px;">{top_table}</div>
          </details>
          <details style="margin-top:10px;">
            <summary style="cursor:pointer;">Representative text excerpts</summary>
            <div style="margin-top:8px;">{reps_html}</div>
          </details>
          <div style="margin-top:14px;"><a href="#top">Back to top</a></div>
        </section>
        """
        sections.append(section_html)

    toc_items = [f'<li><a href="#topic-{int(r["topic_id"])}">Topic {int(r["topic_id"])} — {r["topic_name"]}</a></li>' for _, r in topics_df.iterrows()]
    toc_html = f"<ol>{''.join(toc_items)}</ol>"
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M")

    all_html = f"""<!doctype html><meta charset="utf-8">
    <title>Topics (Text Baseline)</title>
    <div id="top" style="font-family:system-ui,-apple-system,Segoe UI,Roboto,Arial;padding:18px;max-width:1220px;margin:0 auto;">
      <h1 style="margin:0 0 8px 0;">Topics Overview (Text Baseline)</h1>
      <p style="color:#666;margin-top:0;">All topics estimated from text; images are re-ranked per topic with CLIP. Click a speech ID to open its source if a URL exists.</p>
      <div style="background:#fafafa;border:1px solid #eee;border-radius:8px;padding:12px;margin:12px 0;">
        <h3 style="margin:0 0 8px 0;">Contents</h3>
        {toc_html}
      </div>
      {''.join(sections)}
      <hr style="margin:24px 0;">
      <div style="color:#888;font:12px system-ui;">
        Generated: {stamp} • Text model: {TEXT_MODEL} • Image model: {CLIP_MODEL} • Min topic size: {MIN_TOPIC_SIZE} • Images/topic: {IMAGES_PER_TOPIC}
      </div>
    </div>"""
    with open(os.path.join(OUTPUT_DIR, "topics_all_in_one.html"), "w", encoding="utf-8") as f:
        f.write(all_html)
    log.info("Saved topics_all_in_one.html")

    # ---- Image→topic CSV (scores for images actually ranked in HTML get rank)
    img_rows = []
    for i in tqdm(range(len(df)), desc="Image→topic scoring (manifested)"):
        sid = df.loc[i, "ID"]
        assigned_t = int(topics[i])
        row_imgs = img_manifest[img_manifest["ID"] == sid]
        if row_imgs.empty or assigned_t not in prompt_vecs:
            continue
        pvec = prompt_vecs[assigned_t]
        link = None
        if "url" in df.columns and isinstance(df.loc[i, "url"], str) and df.loc[i, "url"].strip():
            link = df.loc[i, "url"]
        elif URL_TEMPLATE:
            link = URL_TEMPLATE.format(id=sid)
        for _, r in row_imgs.head(MAX_IMAGES_PER_SPEECH).iterrows():
            vec = np.load(r["vec_path"]).astype(np.float32)
            vec = vec / (np.linalg.norm(vec) + 1e-12)
            score = float(np.dot(pvec, vec))
            img_rows.append({"ID": sid, "image_path": r["image_path"], "topic": assigned_t, "score": score, "url": link})
    img_map = pd.DataFrame(img_rows)

    # mark the 10 images per topic shown in HTML with ranks
    for tid in topics_df["topic_id"].astype(int).tolist():
        ranked = rerank_images_for_topic(tid, IMAGES_PER_TOPIC, cap=300)
        for rank, (score, sid, p, link) in enumerate(ranked, start=1):
            mask = (img_map["ID"] == sid) & (img_map["image_path"] == p) & (img_map["topic"] == tid)
            img_map.loc[mask, "rank_in_topic"] = rank
    img_map.to_csv(os.path.join(OUTPUT_DIR, "image_topk_topics.csv"), index=False)
    log.info("Saved image_topk_topics.csv")

    tpl = topics_df.copy()
    tpl["topic_label"] = ""
    tpl["topic_group"] = ""
    tpl = tpl[["topic_id","topic_name","topic_label","topic_group","keywords","count"]]
    tpl.to_csv(os.path.join(OUTPUT_DIR, "topic_labels_template.csv"), index=False)
    log.info("Saved topic_labels_template.csv")

    topic_model.save(os.path.join(OUTPUT_DIR, "topic_model_text_only"))
    run_log = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "csv_path": CSV_PATH,
        "image_root": IMAGE_ROOT,
        "rows_total": int(len(df)),
        "device": DEVICE,
        "text_model": TEXT_MODEL,
        "clip_model": CLIP_MODEL,
        "chunk": {"body": CHUNK_BODY, "stride": CHUNK_STRIDE, "max_chunks_per_doc": MAX_CHUNKS_PER_DOC},
        "batch": {"text": BATCH_TEXT, "images": BATCH_IMAGES},
        "min_topic_size": MIN_TOPIC_SIZE,
        "images_per_topic": IMAGES_PER_TOPIC,
        "vectorizer": {"stop_words": "english", "ngram_range": [1, 2], "min_df": 3}
    }
    with open(os.path.join(OUTPUT_DIR, "run_log.json"), "w", encoding="utf-8") as f:
        json.dump(run_log, f, ensure_ascii=False, indent=2)
    log.info("DONE.")

main()

## K-sweep (EN only): reduce_topics() from 200→10 and compute composite score + plot

**Run this cell only for English** dataset (Kremlin_EN or MID_EN).
For RU datasets, we will **skip** K-sweep and directly reduce to the chosen K (same K as its EN partner).

- It Loads the **base** model, Gets initial `(topics0, probs0)` once.
- Runs a **descending** K list: `200, 190, …, 10` (auto-clamped to `<= base_k`).
- Uses **monotone reductions** and Saves
  - `k_sweep_metrics.csv`
  - `k_scree_plot.png`
  - prints best-K by composite score

## Pick final K for this run (EN from K-sweep, RU from its paired EN)

we got this K value for the english datasets, which we also use for the russian versions.

  - Kremlin_RU, Kremlin_EN, the selected K value is 89
  - Mid_RU, Mid_EN, the selected K value is 32

We have hardcoded these two numbers as Kremlin(89) and MID(32), as we got these values from the knee plot.



In [ ]:
# K SWEEP 40..185 (step 10 + 185), c_npmi coherence, NO refits, NO NaNs ---

# Paths (edit if yours differ)
OUTPUT_DIR = "/content/drive/MyDrive/English/csv files/Final CSV Files/bertopic_text_baseline_outputs"
CSV_PATH   = "/content/drive/MyDrive/English/csv files/Final CSV Files/kremlin_transcripts_corrected_final.csv"
EMB_PATH   = OUTPUT_DIR + "/text_embeddings.npz"
MODEL_IN   = OUTPUT_DIR + "/topic_model_text_only"   # your original, unreduced model

# Mount Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

import os, re, copy, random, inspect
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import minmax_scale
from sklearn.metrics.pairwise import cosine_similarity

from gensim.utils import simple_preprocess
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from gensim.parsing.preprocessing import STOPWORDS

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer

# -------- Load artifacts --------
if not os.path.exists(CSV_PATH): raise FileNotFoundError(CSV_PATH)
if not os.path.exists(EMB_PATH): raise FileNotFoundError(EMB_PATH)

df   = pd.read_csv(CSV_PATH, encoding="utf-8", dtype=str, low_memory=False)
docs = df["full_text"].astype(str).tolist()
emb  = np.load(EMB_PATH)["X"].astype(np.float32)

base_model = BERTopic.load(MODEL_IN)  # must be the original, unreduced model

# -------- Repro seeds + lock UMAP RNG --------
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED); random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
except Exception:
    pass
try:
    if hasattr(base_model, "umap_model") and hasattr(base_model.umap_model, "random_state"):
        base_model.umap_model.random_state = SEED
except Exception:
    pass

# -------- Tokenize once for coherence --------
tokens = [simple_preprocess(re.sub(r"\s+"," ", t.replace("\xa0"," ")).strip(),
                            deacc=True, min_len=2, max_len=30) for t in docs]
dictionary = Dictionary(tokens)

TOP_WORDS = 10
MIN_DOCS_PER_TOPIC = 2

def topic_words_fallback(model, tid, member_idx, topn=TOP_WORDS):
    """Use model top-words; if sparse/empty, back off to token freq from that topic’s docs."""
    pairs = model.get_topic(tid) or []
    words = [w for w,_ in pairs[:topn] if isinstance(w, str)]
    if len(words) >= max(3, min(topn, 5)):
        return words[:topn]
    from collections import Counter
    ctr = Counter()
    for i in member_idx:
        ctr.update([w for w in tokens[i] if w not in STOPWORDS])
    fallback = [w for w,_ in ctr.most_common(300) if len(w) >= 2 and not w.isdigit()]
    seen = set(words)
    words = words + [w for w in fallback if w not in seen]
    return words[:topn]

def coherence_npmi_safe(wordlists):
    try:
        if not wordlists: return 0.0
        cm = CoherenceModel(topics=wordlists, texts=tokens, dictionary=dictionary,
                            coherence="c_npmi", processes=1)
        v = float(cm.get_coherence())
        return 0.0 if not np.isfinite(v) else v
    except Exception:
        return 0.0

def metrics_at(model, assignments, topn=TOP_WORDS):
    # Collect members per topic
    all_tids = [t for t in model.get_topics().keys() if t != -1]
    members = {tid: [] for tid in all_tids}
    for i, t in enumerate(assignments):
        if t in members: members[t].append(i)
    tids = [tid for tid in all_tids if len(members[tid]) >= MIN_DOCS_PER_TOPIC]
    if not tids:
        return dict(coherence=0.0, diversity=0.0, compactness=0.0, separation=0.0)

    # Top-words with fallback → avoids NaNs in coherence
    wlists = []
    for tid in tids:
        ws = topic_words_fallback(model, tid, members[tid], topn=topn)
        if ws: wlists.append(ws)

    coherence = coherence_npmi_safe(wlists)

    # Diversity = unique / total
    flat = [w for ws in wlists for w in ws]
    diversity = (len(set(flat))/len(flat)) if flat else 0.0

    # Compactness = mean cos(doc, centroid)
    cents = {}
    for tid in tids:
        idx = members[tid]
        if idx:
            cents[tid] = emb[idx].mean(axis=0)
    sims = []
    for tid in tids:
        idx = members[tid]
        if not idx or tid not in cents: continue
        sims.extend(cosine_similarity(emb[idx], cents[tid].reshape(1,-1)).ravel().tolist())
    compactness = float(np.mean(sims)) if sims else 0.0

    # Separation = mean (1 - cos) between centroids
    if len(cents) > 1:
        C = np.vstack([cents[tid] for tid in cents])
        S = cosine_similarity(C); D = 1.0 - S
        separation = float(np.mean(D[np.triu_indices(D.shape[0], 1)]))
    else:
        separation = 0.0

    return dict(coherence=coherence, diversity=diversity, compactness=compactness, separation=separation)

# -------- Version-agnostic reduce_topics wrapper --------
def reduce_topics_compat(m, docs, topics, probs, k):
    fn = m.reduce_topics
    sig = inspect.signature(fn)
    params = list(sig.parameters.keys())
    try:
        if {"docs","topics","probabilities","nr_topics"}.issubset(params):
            out = fn(docs=docs, topics=topics, probabilities=probs, nr_topics=int(k))
        elif params[:4] == ["docs","topics","probabilities","nr_topics"]:
            out = fn(docs, topics, probs, int(k))
        elif {"docs","nr_topics"}.issubset(params):
            out = fn(docs=docs, nr_topics=int(k))
        elif len(params) >= 2 and params[1] == "nr_topics":
            out = fn(docs, int(k))
        else:
            out = fn(docs, topics, probs, int(k))
    except TypeError:
        try:
            out = fn(docs, topics, probs, int(k))
        except Exception:
            out = fn(docs, nr_topics=int(k))
    if isinstance(out, tuple):
        if   len(out) == 3: mk, tk, pk = out
        elif len(out) == 2: mk, tk = out; pk = None
        else: mk = out[0]; tk, pk = mk.transform(docs, embeddings=emb)
    else:
        mk = out; tk, pk = mk.transform(docs, embeddings=emb)
    return mk, tk, pk

# -------- Exact K list: 40, 50, 60, …, 180, 185 (clamped to base_k) --------
REQ = list(range(40, 190, 10)) + [185]
REQ = sorted(set(REQ))
base_k = len([t for t in base_model.get_topics().keys() if t != -1])
K_LIST_DESC = sorted([k for k in REQ if k <= base_k], reverse=True)
print(f"base_k={base_k} | K targets (desc): {K_LIST_DESC}")

# -------- Sweep (monotone reductions; no refits) --------
seed_model = copy.deepcopy(base_model)
topics0, probs0 = seed_model.transform(docs, embeddings=emb)

rows = []
curr_m, curr_t, curr_p = seed_model, topics0, probs0
curr_k = base_k

for k in K_LIST_DESC:
    if k == curr_k:
        mk, tk, pk = curr_m, curr_t, curr_p
    else:
        mk, tk, pk = reduce_topics_compat(curr_m, docs, curr_t, curr_p, k)
    metr = metrics_at(mk, tk)
    rows.append({"K": int(k), **metr})
    curr_m, curr_t, curr_p = mk, tk, pk
    curr_k = k
    print(f"K={k} → c_npmi={metr['coherence']:.4f}  div={metr['diversity']:.4f}  comp={metr['compactness']:.4f}  sep={metr['separation']:.4f}")

res = pd.DataFrame(rows).sort_values("K")  # ascending in CSV

# -------- Score with your weights --------
W = {"coherence":0.40, "diversity":0.20, "compactness":0.25, "separation":0.15}
def norm(x):
    x = np.asarray(x, float)
    return np.zeros_like(x) if np.allclose(x.max(), x.min()) else minmax_scale(x)

res["coherence_norm"]   = norm(res["coherence"])
res["diversity_norm"]   = norm(res["diversity"])
res["compactness_norm"] = norm(res["compactness"])
res["separation_norm"]  = norm(res["separation"])
res["score"] = (W["coherence"]  * res["coherence_norm"] +
                W["diversity"]  * res["diversity_norm"] +
                W["compactness"]* res["compactness_norm"] +
                W["separation"] * res["separation_norm"])

# -------- Save & plot --------
OUT_DIR = os.path.join(OUTPUT_DIR, "V2_reduced_topics"); os.makedirs(OUT_DIR, exist_ok=True)
METRICS = os.path.join(OUT_DIR, "k_sweep_metrics.csv")
PLOT    = os.path.join(OUT_DIR, "k_scree_plot.png")

res.to_csv(METRICS, index=False)
print("Saved metrics:", METRICS)

plt.figure(figsize=(10.5, 6.0))
Ks = res["K"].values
plt.plot(Ks, res["coherence_norm"],   marker="o", label="Coherence (c_npmi, norm)")
plt.plot(Ks, res["diversity_norm"],   marker="o", label="Diversity (norm)")
plt.plot(Ks, res["compactness_norm"], marker="o", label="Compactness (norm)")
plt.plot(Ks, res["separation_norm"],  marker="o", label="Separation (norm)")
plt.plot(Ks, res["score"],            marker="o", linewidth=3, label="Composite score", alpha=0.85)
bi = int(res["score"].idxmax()); bk, by = int(res.loc[bi,"K"]), float(res.loc[bi,"score"])
plt.scatter([bk], [by], s=160, marker="*", zorder=5, label=f"Chosen K={bk}")
plt.xlabel("Number of topics (K)"); plt.ylabel("Normalized value / score")
plt.title("K sweep: 40, 50, …, 180, 185 (c_npmi, no refits)")
plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout(); plt.savefig(PLOT, dpi=160); plt.close()
print("Scree plot:", PLOT)

# Show the table nicely
print(res[["K","coherence","diversity","compactness","separation","score"]].to_string(index=False))

## Build final reduced BERTopic model (no refits) + save core outputs

**Goal:** Reduce the already-fit base model down to 89(for kremlin), and 32(for mid) using `reduce_topics()` (monotone reduce, no re-embedding), then export:
- `topics.csv` (topic_id, keywords, count)
- `speech_topk_topics.csv` (per speech: top_topic, top_prob)
- `doc_topic_probs_long.csv` (heavy: every doc × every topic probability)
- `topic_model_reduced/` (saved reduced BERTopic model)
- Uses `emb` from `text_embeddings.npz` so transform is consistent.

## Build Topics HTML (text topics + image gallery)

**Goal:** Create a single `topics_all_in_one.html` that shows:
- Topic header (topic_id, count, keywords)
- Top speeches (by assigned probability)
- Top images per topic (re-ranked with CLIP prompt vs cached image vectors)
- This does **NOT** re-embed images (uses cached `vec_path`).
- It only embeds the topic text prompts once with CLIP.

**Inputs expected (already created earlier):**
- `FINAL_K_* / topics.csv`
- `FINAL_K_* / speech_topk_topics.csv`
- Image manifest parquet from the image-vector pipeline (ID, image_path, vec_path)
- The same dataset CSV used for BERTopic (for URLs)




In [ ]:
import os, sys, re, json, base64, logging, glob, hashlib
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm import tqdm

from PIL import Image, ImageFile, ImageOps
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from google.colab import drive

CSV_PATH   = "/content/drive/MyDrive/Russian Speech Dataset Project/CSV Files/mid_english.csv"
IMAGE_ROOT = "/content/drive/MyDrive/Russian Speech Dataset Project/Archive/New Files/Mid/English/Scraped Images"
OUTPUT_DIR = "/content/drive/MyDrive/English/csv files/Final CSV Files/bertopic_text_baseline_outputs_jan_rakesh"

# Optional: if URL not in CSV, build from ID
URL_TEMPLATE = ""  # e.g., "https://en.kremlin.ru/events/president/news/{id}"

# Create dirs
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "image_vecs"), exist_ok=True)

TEXT_MODEL  = "sentence-transformers/all-mpnet-base-v2"
CLIP_MODEL  = "clip-ViT-B-32"

# Chunking (cover full text via windows under 512)
CHUNK_BODY          = 448
CHUNK_STRIDE        = 128
MAX_CHUNKS_PER_DOC  = 32

# Batching
BATCH_TEXT   = 32
BATCH_IMAGES = 96

# BERTopic
MIN_TOPIC_SIZE   = 12
TOP_N_TOPICS     = 32 # GIVE THE NUMBER OF SELECTED TOPICS
IMAGES_PER_TOPIC = 10
MAX_IMAGES_PER_SPEECH = 8       # for reranking cost control

# Vectorizer / c-TF-IDF
VECTORIZER = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3)
CTFIDF     = ClassTfidfTransformer(reduce_frequent_words=True)

# Repro
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Logging / device
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("bertopic_text_only")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
log.info(f"Device: {DEVICE}")

# Mount Drive (if not yet)
try:
    drive.mount('/content/drive')
except Exception:
    pass


IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def fail_if_not_utf8(path: str):
    with open(path, "rb") as f:
        data = f.read()
    try:
        data.decode("utf-8")
    except UnicodeDecodeError:
        log.error(f"CSV is not UTF-8: {path}")
        sys.exit(1)

def read_csv_utf8(path: str) -> pd.DataFrame:
    fail_if_not_utf8(path)
    df = pd.read_csv(path, encoding="utf-8")

    df["ID"] = df["id"].astype(str)
    df["full_text"] = df["full_text"].astype(str)
    return df

def simple_text_clean(t: str) -> str:
    t = t.replace("\xa0", " ")
    t = re.sub(r"<[^>]+>", " ", t)     # strip HTML
    t = re.sub(r"\s+", " ", t).strip()
    return t

def list_images_for_id(speech_id) -> list[str]:
    """Folder-per-ID: IMAGE_ROOT/<ID>/*"""
    folder = os.path.join(IMAGE_ROOT, str(speech_id).strip())
    paths = []
    if os.path.isdir(folder):
        for p in sorted(glob.glob(os.path.join(folder, "*"))):
            if os.path.splitext(p.lower())[1] in IMG_EXTS and os.path.exists(p):
                paths.append(p)
    return paths

def split_images_cell(cell: str) -> list[str]:
    paths = []
    for part in str(cell).split("||"):
        part = part.strip()
        if not part:
            continue
        cand = os.path.join(IMAGE_ROOT, part) if not os.path.isabs(part) else part
        if os.path.exists(cand):
            paths.append(cand)
    return paths

def prefer_images_for_row(row) -> list[str]:
    # Prefer an explicit "image_filenames" column if present; otherwise folder-per-ID
    if "image_filenames" in row and str(row["image_filenames"]).strip():
        paths = split_images_cell(row["image_filenames"])
        if paths:
            return paths
    return list_images_for_id(row["ID"])

def load_resize_image(path: str, size=(512, 512)):
    try:
        resample = getattr(Image, "LANCZOS", Image.BICUBIC)
        im = Image.open(path).convert("RGB")
        im = ImageOps.exif_transpose(im)
        im = im.resize(size, resample)
        return im
    except Exception as e:
        log.warning(f"Image load failed '{path}': {e}")
        return None

def encode_png_base64(im: Image.Image, size=(384, 384)) -> str:
    im = ImageOps.exif_transpose(im).convert("RGB")
    resample = getattr(Image, "LANCZOS", Image.BICUBIC)
    im = im.resize(size, resample).copy()
    buf = BytesIO()
    im.save(buf, format="PNG", optimize=False)   # PNG avoids Pillow JPEG fileno issue
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def l2norm(mat: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(mat, axis=1, keepdims=True)
    n[n == 0.0] = 1.0
    return mat / n

def build_chunk_texts(text: str, tokenizer: AutoTokenizer) -> list[str]:
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    if not ids:
        return []
    chunks = []
    for start in range(0, len(ids), CHUNK_STRIDE):
        window = ids[start:start + CHUNK_BODY]
        if not window:
            break
        ch = tokenizer.decode(window, skip_special_tokens=True).strip()
        if ch:
            chunks.append(ch)
        if len(chunks) >= MAX_CHUNKS_PER_DOC:
            break
    if not chunks:
        chunks = [text[:2000]]
    return chunks

def topic_prompt(keyword_string: str) -> str:
    return f"news photo of {keyword_string}"

def vec_path_for_image(image_path: str) -> str:
    # stable filename in OUTPUT_DIR/image_vecs/
    h = hashlib.md5(image_path.encode("utf-8")).hexdigest()
    base = f"{h}.npy"
    return os.path.join(OUTPUT_DIR, "image_vecs", base)


def main():
    # ---- Load CSV
    log.info("Reading CSV…")
    df = read_csv_utf8(CSV_PATH).copy()
    df["full_text"] = df["full_text"].map(simple_text_clean)

    # ---- Resolve images (but DO NOT drop rows without images)
    log.info("Align images…")
    all_paths = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Align images"):
        all_paths.append(prefer_images_for_row(row))
    df["all_image_paths"] = all_paths

    # ---- Text model and tokenizer
    log.info("Loading text model + tokenizer…")
    text_model = SentenceTransformer(TEXT_MODEL, device=DEVICE)
    tokenizer  = AutoTokenizer.from_pretrained(TEXT_MODEL, use_fast=True)

    # ---- Chunk + embed text (cache all to Drive)
    log.info("Chunking…")
    chunk_lists = []
    for t in tqdm(df["full_text"].tolist(), total=len(df), desc="Chunking"):
        chunk_lists.append(build_chunk_texts(t, tokenizer))

    log.info("Encoding text chunks…")
    dim = text_model.get_sentence_embedding_dimension()
    text_doc_vecs = np.zeros((len(df), dim), dtype=np.float32)

    work = []
    for i, chunks in enumerate(chunk_lists):
        for ch in chunks:
            work.append((i, ch))
    results_per_doc = [[] for _ in range(len(df))]

    with tqdm(total=len(work), desc="Encode text chunks") as bar:
        cursor = 0
        while cursor < len(work):
            end = min(cursor + BATCH_TEXT, len(work))
            owners = [work[j][0] for j in range(cursor, end)]
            texts  = [work[j][1] for j in range(cursor, end)]
            vecs = text_model.encode(
                texts, batch_size=len(texts), device=DEVICE,
                show_progress_bar=False, convert_to_numpy=True,
                normalize_embeddings=False
            )
            for o, v in zip(owners, vecs):
                results_per_doc[o].append(v.astype(np.float32))
            cursor = end
            bar.update(len(texts))

    for i in range(len(df)):
        if results_per_doc[i]:
            text_doc_vecs[i] = np.mean(results_per_doc[i], axis=0)
        else:
            v = text_model.encode([df.loc[i,"full_text"][:2000]], device=DEVICE,
                                  show_progress_bar=False, convert_to_numpy=True)
            text_doc_vecs[i] = v[0].astype(np.float32)

    # Save text embeddings + doc index (for future reuse keyed by ID)
    np.savez_compressed(os.path.join(OUTPUT_DIR, "text_embeddings.npz"), X=text_doc_vecs)
    pd.DataFrame({"row_idx": np.arange(len(df)), "ID": df["ID"]}).to_csv(
        os.path.join(OUTPUT_DIR, "doc_index.csv"), index=False
    )
    log.info("Saved text_embeddings.npz and doc_index.csv")

    # ---- Fit BERTopic (text-only)
    log.info("Fitting BERTopic (text-only)…")
    topic_model = BERTopic(
        embedding_model=None,
        vectorizer_model=VECTORIZER,
        ctfidf_model=CTFIDF,
        min_topic_size=MIN_TOPIC_SIZE,
        nr_topics=None,
        calculate_probabilities=True,
        low_memory=True,
        verbose=True
    )
    docs = df["full_text"].tolist()
    topics, probs = topic_model.fit_transform(docs, embeddings=text_doc_vecs)

    # ---- Topic info (ALL non -1 topics)
    info = topic_model.get_topic_info()
    info = info[info["Topic"] != -1].sort_values("Count", ascending=False).copy()

    def kw_for(tid: int) -> str:
        pairs = topic_model.get_topic(int(tid)) or []
        return ", ".join([w for (w, _) in pairs])

    info["Keywords"] = info["Topic"].map(kw_for)
    topics_df = info.rename(columns={"Topic":"topic_id","Name":"topic_name","Count":"count","Keywords":"keywords"})
    topics_df = topics_df[["topic_id","topic_name","keywords","count"]]
    topics_df.to_csv(os.path.join(OUTPUT_DIR,"topics.csv"), index=False)
    log.info("Saved topics.csv")

    # ---- Speech → top topic mapping (ALL rows, even if prob low)
    probs_np = np.array(probs) if probs is not None else None
    topic2kw = {int(r["topic_id"]): r["keywords"] for _, r in topics_df.iterrows()}

    speech_rows = []
    for i in range(len(df)):
        sid = df.loc[i, "ID"]
        assigned_tid = int(topics[i])
        assigned_prob = None
        if probs_np is not None and probs_np.ndim == 2 and i < probs_np.shape[0]:
            assigned_prob = float(np.max(probs_np[i]))
        url = None
        if "url" in df.columns and isinstance(df.loc[i, "url"], str) and df.loc[i, "url"].strip():
            url = df.loc[i,"url"]
        elif URL_TEMPLATE:
            url = URL_TEMPLATE.format(id=sid)
        speech_rows.append({
            "ID": sid,
            "top_topic": assigned_tid,
            "top_prob": assigned_prob,
            "topic_keywords": topic2kw.get(assigned_tid, ""),
            "url": url
        })
    speech_topk = pd.DataFrame(speech_rows)
    speech_topk.to_csv(os.path.join(OUTPUT_DIR, "speech_topk_topics.csv"), index=False)
    log.info("Saved speech_topk_topics.csv")

    # ---- Representative docs HTML (preview)
    rows = []
    for tid in topics_df["topic_id"].astype(int).tolist():
        reps = topic_model.get_representative_docs(int(tid)) or []
        for doc in reps[:3]:
            prev = (doc[:600] + "…") if len(doc) > 600 else doc
            rows.append({"topic_id": int(tid), "preview": prev})
    rep_df = pd.DataFrame(rows)
    with open(os.path.join(OUTPUT_DIR, "representative_docs.html"), "w", encoding="utf-8") as f:
        f.write(rep_df.to_html(index=False, escape=True))
    log.info("Saved representative_docs.html")

    # ---- Image embeddings (cache by ID+path to Drive)
    log.info("Embedding images with CLIP (cache & reuse)…")
    clip_model = SentenceTransformer(CLIP_MODEL, device=DEVICE)

    # We will create/append a manifest: (ID, image_path, vec_npy_path)
    manifest_path = os.path.join(OUTPUT_DIR, "image_embeddings.parquet")
    if os.path.exists(manifest_path):
        img_manifest = pd.read_parquet(manifest_path)
    else:
        img_manifest = pd.DataFrame(columns=["ID","image_path","vec_path"])

    existing = set((row["ID"], row["image_path"]) for _, row in img_manifest.iterrows())
    new_rows = []

    # Load representative first image per speech for BERTopic visual rep (optional later) and
    # also prepare all image paths for per-topic reranking
    rep_imgs = [paths[0] if len(paths)>0 else None for paths in df["all_image_paths"]]

    # Batch-embed ONLY missing images, save per-image vector to .npy file in OUTPUT_DIR/image_vecs/
    to_embed = []
    id_for = []
    for i in range(len(df)):
        sid = df.loc[i, "ID"]
        for p in df.loc[i,"all_image_paths"][:MAX_IMAGES_PER_SPEECH]:
            key = (sid, p)
            if key not in existing:
                to_embed.append(p)
                id_for.append(sid)

    # Split into chunks to not overflow memory
    for start in tqdm(range(0, len(to_embed), BATCH_IMAGES), desc="Image batches"):
        batch_paths = to_embed[start:start+BATCH_IMAGES]
        images = [load_resize_image(p) for p in batch_paths]
        keep = [(p, im) for p, im in zip(batch_paths, images) if im is not None]
        if not keep:
            continue
        batch_paths2, ims2 = zip(*keep)
        vecs = clip_model.encode(list(ims2), batch_size=min(BATCH_IMAGES, len(ims2)),
                                 device=DEVICE, show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
        for pth, vec in zip(batch_paths2, vecs):
            vecf = vec_path_for_image(pth)
            np.save(vecf, vec)
            sid = id_for[to_embed.index(pth)]  # safe since unique in 'to_embed'
            new_rows.append({"ID": sid, "image_path": pth, "vec_path": vecf})

    if new_rows:
        img_manifest = pd.concat([img_manifest, pd.DataFrame(new_rows)], ignore_index=True)
        img_manifest.drop_duplicates(subset=["ID","image_path"], inplace=True)
        img_manifest.to_parquet(manifest_path, index=False)
    log.info(f"Saved/updated image_embeddings.parquet with {len(img_manifest)} rows")

    # ---- Build topic prompts (from keywords) and embed once
    kw_map = {int(r["topic_id"]): r["keywords"] for _, r in topics_df.iterrows()}
    prompt_vecs = []
    prompt_topic_ids = []
    prompts = []
    for tid, kws in kw_map.items():
        prompt = topic_prompt(kws)
        prompts.append(prompt)
        prompt_topic_ids.append(int(tid))
    if prompts:
        pvecs = clip_model.encode(prompts, device=DEVICE, show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
        pvecs = l2norm(pvecs)
        prompt_vecs = {tid: pvecs[i] for i, tid in enumerate(prompt_topic_ids)}
    else:
        prompt_vecs = {}

    # ---- Rerank images per topic (cosine(prompt,image))
    def rerank_images_for_topic(tid: int, topk: int = IMAGES_PER_TOPIC, cap: int = 300):
        if tid not in prompt_vecs:
            return []
        pvec = prompt_vecs[tid]
        cand = []

        # collect members (speech indices) for tid by max-prob ranking
        probs_np_local = np.array(probs) if probs is not None else None
        members = []
        for i, t in enumerate(topics):
            if t == tid:
                pr = float(np.max(probs_np_local[i])) if probs_np_local is not None and probs_np_local.ndim == 2 else 1.0
                members.append((pr, i))
        members.sort(key=lambda x: -x[0])

        # walk members and their images
        for pr, row_idx in members:
            sid = df.loc[row_idx, "ID"]
            row_imgs = img_manifest[img_manifest["ID"] == sid]
            if row_imgs.empty:
                continue
            for _, r in row_imgs.head(MAX_IMAGES_PER_SPEECH).iterrows():
                vec = np.load(r["vec_path"]).astype(np.float32)
                vec = vec / (np.linalg.norm(vec) + 1e-12)
                score = float(np.dot(pvec, vec))
                link = None
                if "url" in df.columns and isinstance(df.loc[row_idx, "url"], str) and df.loc[row_idx, "url"].strip():
                    link = df.loc[row_idx, "url"]
                elif URL_TEMPLATE:
                    link = URL_TEMPLATE.format(id=sid)
                cand.append((score, sid, r["image_path"], link))
                if len(cand) >= cap:
                    break
            if len(cand) >= cap:
                break
        cand.sort(key=lambda x: -x[0])
        return cand[:topk]

    # ---- Build big HTML with 10 images/topic + top speeches + reps
    def make_tile(path, sid, link, thumb_w=200):
        im = load_resize_image(path, size=(384,384))
        if im is None: return ""
        b64 = encode_png_base64(im, size=(384, 384))
        link_html = f' — <a href="{link}" target="_blank">open</a>' if link else ""
        return (
            f'<figure style="width:{thumb_w}px;margin:6px;">'
            f'  <img loading="lazy" src="data:image/png;base64,{b64}" '
            f'       style="width:{thumb_w}px;border:1px solid #ddd;border-radius:6px;">'
            f'  <figcaption style="font:12px/1.35 system-ui;color:#444;margin-top:4px;">ID <b>{sid}</b>{link_html}</figcaption>'
            f'</figure>'
        )

    # Precompute topic → members (top speeches table)
    topic_members = {}
    probs_np = np.array(probs) if probs is not None else None
    for i, t in enumerate(topics):
        if t == -1:
            continue
        pr = float(np.max(probs_np[i])) if probs_np is not None and probs_np.ndim == 2 else 1.0
        topic_members.setdefault(int(t), []).append((pr, i))
    for t in topic_members:
        topic_members[t].sort(key=lambda x: -x[0])

    sections = []
    for _, r in topics_df.iterrows():
        tid   = int(r["topic_id"])
        name  = r["topic_name"]
        count = int(r["count"])
        keys  = r["keywords"]

        ranked = rerank_images_for_topic(tid, IMAGES_PER_TOPIC, cap=300)
        tiles  = [make_tile(p, sid, link) for (score, sid, p, link) in ranked]
        gallery = ''.join(tiles) if tiles else '<div style="color:#777;">No images available</div>'

        # Top speeches (first 6)
        rows_html = []
        for pr, idx in topic_members.get(tid, [])[:6]:
            sid = df.loc[idx, "ID"]
            if "url" in df.columns and isinstance(df.loc[idx, "url"], str) and df.loc[idx, "url"].strip():
                url_html = f'<a href="{df.loc[idx, "url"]}" target="_blank">{sid}</a>'
            elif URL_TEMPLATE:
                url_html = f'<a href="{URL_TEMPLATE.format(id=sid)}" target="_blank">{sid}</a>'
            else:
                url_html = f"{sid}"
            rows_html.append(
                f"<tr><td style='padding:6px 10px;border-bottom:1px solid #eee'>{url_html}</td>"
                f"<td style='padding:6px 10px;border-bottom:1px solid #eee;text-align:right'>{pr:.3f}</td></tr>"
            )
        top_table = (
            "<table style='border-collapse:collapse;font:13px system-ui;'>"
            "<thead><tr><th style='text-align:left;padding:6px 10px;border-bottom:1px solid #ddd'>Speech ID</th>"
            "<th style='text-align:right;padding:6px 10px;border-bottom:1px solid #ddd'>Assigned prob</th></tr></thead>"
            f"<tbody>{''.join(rows_html)}</tbody></table>"
        )

        # Representative text excerpts
        reps = topic_model.get_representative_docs(tid) or []
        rep_blocks = []
        for doc in reps[:3]:
            prev = (doc[:600] + "…") if len(doc) > 600 else doc
            rep_blocks.append(f"<div style='margin:8px 0;padding:10px;background:#fafafa;border:1px solid #eee;border-radius:6px;'>{prev}</div>")
        reps_html = ''.join(rep_blocks) if rep_blocks else "<div style='color:#777;'>No previews</div>"

        section_html = f"""
        <section id="topic-{tid}" style="margin:24px 0 36px 0;padding-top:24px;border-top:2px solid #f2f2f2;">
          <h2 style="margin:0 0 6px 0;">Topic {tid} — {name}</h2>
          <div style="color:#666;margin:0 0 12px 0;">Count: <b>{count}</b></div>
          <div style="color:#444;margin:0 0 14px 0;"><b>Keywords:</b> {keys}</div>
          <div style="display:flex;flex-wrap:wrap;gap:6px;">{gallery}</div>
          <details style="margin-top:14px;">
            <summary style="cursor:pointer;">Top speeches (first 6)</summary>
            <div style="margin-top:10px;">{top_table}</div>
          </details>
          <details style="margin-top:10px;">
            <summary style="cursor:pointer;">Representative text excerpts</summary>
            <div style="margin-top:8px;">{reps_html}</div>
          </details>
          <div style="margin-top:14px;"><a href="#top">Back to top</a></div>
        </section>
        """
        sections.append(section_html)

    toc_items = [f'<li><a href="#topic-{int(r["topic_id"])}">Topic {int(r["topic_id"])} — {r["topic_name"]}</a></li>' for _, r in topics_df.iterrows()]
    toc_html = f"<ol>{''.join(toc_items)}</ol>"
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M")

    all_html = f"""<!doctype html><meta charset="utf-8">
    <title>Topics (Text Baseline)</title>
    <div id="top" style="font-family:system-ui,-apple-system,Segoe UI,Roboto,Arial;padding:18px;max-width:1220px;margin:0 auto;">
      <h1 style="margin:0 0 8px 0;">Topics Overview (Text Baseline)</h1>
      <p style="color:#666;margin-top:0;">All topics estimated from text; images are re-ranked per topic with CLIP. Click a speech ID to open its source if a URL exists.</p>
      <div style="background:#fafafa;border:1px solid #eee;border-radius:8px;padding:12px;margin:12px 0;">
        <h3 style="margin:0 0 8px 0;">Contents</h3>
        {toc_html}
      </div>
      {''.join(sections)}
      <hr style="margin:24px 0;">
      <div style="color:#888;font:12px system-ui;">
        Generated: {stamp} • Text model: {TEXT_MODEL} • Image model: {CLIP_MODEL} • Min topic size: {MIN_TOPIC_SIZE} • Images/topic: {IMAGES_PER_TOPIC}
      </div>
    </div>"""
    with open(os.path.join(OUTPUT_DIR, "topics_all_in_one.html"), "w", encoding="utf-8") as f:
        f.write(all_html)
    log.info("Saved topics_all_in_one.html")

    # ---- Image→topic CSV (scores for images actually ranked in HTML get rank)
    img_rows = []
    for i in tqdm(range(len(df)), desc="Image→topic scoring (manifested)"):
        sid = df.loc[i, "ID"]
        assigned_t = int(topics[i])
        row_imgs = img_manifest[img_manifest["ID"] == sid]
        if row_imgs.empty or assigned_t not in prompt_vecs:
            continue
        pvec = prompt_vecs[assigned_t]
        link = None
        if "url" in df.columns and isinstance(df.loc[i, "url"], str) and df.loc[i, "url"].strip():
            link = df.loc[i, "url"]
        elif URL_TEMPLATE:
            link = URL_TEMPLATE.format(id=sid)
        for _, r in row_imgs.head(MAX_IMAGES_PER_SPEECH).iterrows():
            vec = np.load(r["vec_path"]).astype(np.float32)
            vec = vec / (np.linalg.norm(vec) + 1e-12)
            score = float(np.dot(pvec, vec))
            img_rows.append({"ID": sid, "image_path": r["image_path"], "topic": assigned_t, "score": score, "url": link})
    img_map = pd.DataFrame(img_rows)

    # mark the 10 images per topic shown in HTML with ranks
    for tid in topics_df["topic_id"].astype(int).tolist():
        ranked = rerank_images_for_topic(tid, IMAGES_PER_TOPIC, cap=300)
        for rank, (score, sid, p, link) in enumerate(ranked, start=1):
            mask = (img_map["ID"] == sid) & (img_map["image_path"] == p) & (img_map["topic"] == tid)
            img_map.loc[mask, "rank_in_topic"] = rank
    img_map.to_csv(os.path.join(OUTPUT_DIR, "image_topk_topics.csv"), index=False)
    log.info("Saved image_topk_topics.csv")

    tpl = topics_df.copy()
    tpl["topic_label"] = ""
    tpl["topic_group"] = ""
    tpl = tpl[["topic_id","topic_name","topic_label","topic_group","keywords","count"]]
    tpl.to_csv(os.path.join(OUTPUT_DIR, "topic_labels_template.csv"), index=False)
    log.info("Saved topic_labels_template.csv")

    topic_model.save(os.path.join(OUTPUT_DIR, "topic_model_text_only"))
    run_log = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "csv_path": CSV_PATH,
        "image_root": IMAGE_ROOT,
        "rows_total": int(len(df)),
        "device": DEVICE,
        "text_model": TEXT_MODEL,
        "clip_model": CLIP_MODEL,
        "chunk": {"body": CHUNK_BODY, "stride": CHUNK_STRIDE, "max_chunks_per_doc": MAX_CHUNKS_PER_DOC},
        "batch": {"text": BATCH_TEXT, "images": BATCH_IMAGES},
        "min_topic_size": MIN_TOPIC_SIZE,
        "images_per_topic": IMAGES_PER_TOPIC,
        "vectorizer": {"stop_words": "english", "ngram_range": [1, 2], "min_df": 3}
    }
    with open(os.path.join(OUTPUT_DIR, "run_log.json"), "w", encoding="utf-8") as f:
        json.dump(run_log, f, ensure_ascii=False, indent=2)
    log.info("DONE.")

main()

once we ran the above cell for each of the 4 corpora, we will get the topic id, and the top 1 topic porbability score of each speech belonging to a topic. Our team mates manually created a Topic label and the Group name, we will add these columns to the master csv file for each corpus.


## Create Final “Curated” CSV (Add Curated Text + Curated Image Columns)

Starting from the input CSV, add these final columns and write **one final CSV**:
- `curated_topic_id`
- `curated_text_topic_label`
- `curated_text_topic_group`
- `curated_topic_probability`  *(stores TOP-1 topic probs per speech)*

And for images (aligned to `stored_image_filepaths` order):
- `curated_image_topic_ids`
- `curated_image_topic_labels`
- `curated_image_group_names`
- `curated_image_topic_probabilities`  *(stores TOP-1 topic probs per image)*


In [ ]:
# TOPIC DICTIONARIES (ALL 4 CORPORA)

# KREMLIN ENGLISH
KREMLIN_EN_TOPIC_DICT = {
    0:  {"label": "Russia's neighbours",                    "group": "IR & Bilateral Relations"},
    1:  {"label": "Domestic coalitions",                    "group": "Executive, Legislature, Judicial"},
    2:  {"label": "Russia-West relations",                  "group": "IR & Bilateral Relations"},
    3:  {"label": "Crimea affairs",                         "group": "Domestic Politics"},
    4:  {"label": "WWII commemoration",                     "group": "Military & Security"},
    5:  {"label": "Putin's European allies",                "group": "IR & Bilateral Relations"},
    6:  {"label": "Intergovernmental cooperation",          "group": "IR & Bilateral Relations"},
    7:  {"label": "Russia-China relations",                 "group": "IR & Bilateral Relations"},
    8:  {"label": "Russian Far East",                       "group": "Domestic Politics"},
    9:  {"label": "Domestic business",                      "group": "Domestic Politics"},
    10: {"label": "Energy sector",                          "group": "Domestic Politics"},
    11: {"label": "Russia-Ukraine relations",               "group": "IR & Bilateral Relations"},
    12: {"label": "Olympics",                               "group": "Sports"},
    13: {"label": "Military and defense",                   "group": "Military & Security"},
    14: {"label": "Domestic economy",                       "group": "Domestic Politics"},
    15: {"label": "Healthcare system",                      "group": "Healthcare"},
    16: {"label": "Non-Western allies",                     "group": "IR & Bilateral Relations"},
    17: {"label": "Religion",                               "group": "Culture & Religion"},
    18: {"label": "National award ceremonies",              "group": "Culture & Religion"},
    19: {"label": "East Asian relations",                   "group": "IR & Bilateral Relations"},
    20: {"label": "Education system",                       "group": "Science & Education"},
    21: {"label": "Russia-ASEAN relations",                 "group": "IR & Bilateral Relations"},
    22: {"label": "Russia-Germany relations",               "group": "IR & Bilateral Relations"},
    23: {"label": "Domestic law enforcement",               "group": "Military & Security"},
    24: {"label": "Russia's navy",                          "group": "Military & Security"},
    25: {"label": "Russia-Eurasian cooperation",            "group": "IR & Bilateral Relations"},
    26: {"label": "Security services",                      "group": "Military & Security"},
    27: {"label": "Russia-European relations",              "group": "IR & Bilateral Relations"},
    28: {"label": "Domestic courts",                        "group": "Executive, Legislature, Judicial"},
    29: {"label": "Russia-Georgia relations",               "group": "IR & Bilateral Relations"},
    30: {"label": "Russia's transportation system",         "group": "Domestic Politics"},
    31: {"label": "Russia-Africa relations",                "group": "IR & Bilateral Relations"},
    32: {"label": "Domestic emergency response",            "group": "Domestic Politics"},
    33: {"label": "Russia-India relations",                 "group": "IR & Bilateral Relations"},
    34: {"label": "Russia-Middle Eastern relations",        "group": "IR & Bilateral Relations"},
    35: {"label": "Domestic financial institutions",        "group": "Domestic Politics"},
    36: {"label": "Russia-Israel-Palestine relations",      "group": "IR & Bilateral Relations"},
    37: {"label": "Russia's culture",                       "group": "Culture & Religion"},
    38: {"label": "Scientific developments",                "group": "Science & Education"},
    39: {"label": "Middle-Eastern partnership",             "group": "IR & Bilateral Relations"},
    40: {"label": "Russia-Iran alliance",                   "group": "IR & Bilateral Relations"},
    41: {"label": "Automotive industry",                    "group": "Domestic Politics"},
    42: {"label": "Space industry",                         "group": "Domestic Politics"},
    43: {"label": "Russia-Latin America relations",         "group": "IR & Bilateral Relations"},
    44: {"label": "Chechnya affairs",                       "group": "Domestic Politics"},
    45: {"label": "Terrorism",                              "group": "Military & Security"},
    46: {"label": "Aviation industry",                      "group": "Military & Security"},
    47: {"label": "Russia-Bulgaria-Greece relations",       "group": "IR & Bilateral Relations"},
    48: {"label": "Russia-Scandinavia relations",           "group": "IR & Bilateral Relations"},
    49: {"label": "Domestic agriculture",                   "group": "Domestic Politics"},
    50: {"label": "Russia-Mongolia relations",              "group": "IR & Bilateral Relations"},
    51: {"label": "World Cup",                              "group": "Sports"},
    52: {"label": "Nuclear industry",                       "group": "Military & Security"},
    53: {"label": "New Year's speeches",                    "group": "Culture & Religion"},
    54: {"label": "Environmental protection",               "group": "Domestic Politics"},
    55: {"label": "Elections",                              "group": "Domestic Politics"},
    56: {"label": "Russia-Spain relations",                 "group": "IR & Bilateral Relations"},
    57: {"label": "Hockey",                                 "group": "Sports"},
    58: {"label": "Russia-Egypt relations",                 "group": "IR & Bilateral Relations"},
    59: {"label": "Domestic volunteerism",                  "group": "Domestic Politics"},
    60: {"label": "Construction industry",                  "group": "Domestic Politics"},
    61: {"label": "Domestic unions",                        "group": "Domestic Politics"},
    62: {"label": "Arctic exploration",                     "group": "Military & Security"},
    63: {"label": "Emergency ministry meetings",            "group": "Domestic Politics"},
    64: {"label": "Media",                                  "group": "Domestic Politics"},
    65: {"label": "Investments",                            "group": "Domestic Politics"},
    66: {"label": "Senior citizens",                        "group": "Domestic Politics"},
    67: {"label": "Financial monitoring",                   "group": "Domestic Politics"},
    68: {"label": "Child welfare policy",                   "group": "Domestic Politics"},
    69: {"label": "Narcotics control",                      "group": "Military & Security"},
    70: {"label": "Family affairs",                         "group": "Domestic Politics"},
    71: {"label": "Russia-Brazil relations",                "group": "IR & Bilateral Relations"},
    72: {"label": "Russia's youth",                         "group": "Domestic Politics"},
    73: {"label": "Border control",                         "group": "Military & Security"},
    74: {"label": "Taxation",                               "group": "Domestic Politics"},
    75: {"label": "Northwestern Europe affairs",            "group": "IR & Bilateral Relations"},
    76: {"label": "Women's recognition",                    "group": "Domestic Politics"},
    77: {"label": "Caspian region",                         "group": "Military & Security"},
    78: {"label": "Russia-Afghanistan relations",           "group": "IR & Bilateral Relations"},
    79: {"label": "Russia-UN relations",                    "group": "IR & Bilateral Relations"},
    80: {"label": "Russia-Cuba relations",                  "group": "IR & Bilateral Relations"},
    81: {"label": "Russia-Indonesia relations",             "group": "IR & Bilateral Relations"},
    82: {"label": "Russian Sports",                         "group": "Sports"},
    83: {"label": "Russia's geographical society",          "group": "Domestic Politics"},
    84: {"label": "Domestic tourism",                       "group": "Domestic Politics"},
    85: {"label": "Students",                               "group": "Science & Education"},
    86: {"label": "Agricultural industry",                  "group": "Domestic Politics"},
    87: {"label": "Russia-Cyprus relations",                "group": "IR & Bilateral Relations"},
    88: {"label": "Customs service",                        "group": "Military & Security"},
}

# MID ENGLISH (your 32-topic list)
MID_EN_TOPIC_DICT = {
    0:  {"label": "Ukrainian affairs",                     "group": "Post-Soviet Relations"},
    1:  {"label": "Middle Eastern affairs",                "group": "IR & Bilateral Relations"},
    2:  {"label": "Asian affairs",                         "group": "IR & Bilateral Relations"},
    3:  {"label": "Central Asian affairs",                 "group": "Post-Soviet Relations"},
    4:  {"label": "Compatriots affairs",                   "group": "Post-Soviet Relations"},
    5:  {"label": "South Caucasus affairs",                "group": "Post-Soviet Relations"},
    6:  {"label": "Latin American allies",                 "group": "IR & Bilateral Relations"},
    7:  {"label": "African affairs",                       "group": "IR & Bilateral Relations"},
    8:  {"label": "Russian-Islamic states relations",      "group": "IR & Bilateral Relations"},
    9:  {"label": "Iranian nuclear affairs",               "group": "IR & Bilateral Relations"},
    10: {"label": "Russia's economic development",         "group": "Internal affairs"},
    11: {"label": "Arctic affairs",                        "group": "IR & Bilateral Relations"},
    12: {"label": "Eurasian intergovernemntal cooperation", "group": "Post-Soviet Relations"},
    13: {"label": "Developing nations cooperation",        "group": "IR & Bilateral Relations"},
    14: {"label": "Afghanistan relations",                 "group": "IR & Bilateral Relations"},
    15: {"label": "Cyprus-Greece affairs",                 "group": "IR & Bilateral Relations"},
    16: {"label": "Religion",                              "group": "Internal affairs"},
    17: {"label": "Lavrov's interviews",                   "group": "IR & Bilateral Relations"},
    18: {"label": "Korean affairs",                        "group": "IR & Bilateral Relations"},
    19: {"label": "MENA affairs",                          "group": "IR & Bilateral Relations"},
    20: {"label": "WWII commemoration",                    "group": "Internal affairs"},
    21: {"label": "Russia-Germany relations",              "group": "IR & Bilateral Relations"},
    22: {"label": "MGIMO",                                 "group": "Internal affairs"},
    23: {"label": "UNESCO",                                "group": "IR & Bilateral Relations"},
    24: {"label": "Anti-terrorist cooperation",            "group": "IR & Bilateral Relations"},
    25: {"label": "International sports",                  "group": "Sports"},
    26: {"label": "Russia-EU affairs",                     "group": "IR & Bilateral Relations"},
    27: {"label": "Russia-Vietnam relations",              "group": "IR & Bilateral Relations"},
    28: {"label": "Media",                                 "group": "Internal affairs"},
    29: {"label": "Caspian region",                        "group": "Post-Soviet Relations"},
    30: {"label": "Russia-Poland relations",               "group": "IR & Bilateral Relations"},
    31: {"label": "Russia-Lebanon relations",              "group": "IR & Bilateral Relations"},
}

# KREMLIN RUSSIAN
KREMLIN_RU_TOPIC_DICT = {
    0:  {"label": "Domestic politics",                     "group": "Executive, Legislature, Judicial"},
    1:  {"label": "Russia's neighbours",                   "group": "IR & Bilateral Relations"},
    2:  {"label": "Domestic business",                     "group": "Domestic Politics"},
    3:  {"label": "Russia-West relations",                 "group": "IR & Bilateral Relations"},
    4:  {"label": "Russia-China relations",                "group": "IR & Bilateral Relations"},
    5:  {"label": "Russia-Belarus relations",              "group": "IR & Bilateral Relations"},
    6:  {"label": "Education system",                      "group": "Science & Education"},
    7:  {"label": "Domestic law enforcement",              "group": "Military & Security"},
    8:  {"label": "Russia-Ukraine relations",              "group": "IR & Bilateral Relations"},
    9:  {"label": "Healthcare system",                     "group": "Healthcare"},
    10: {"label": "Russian Sports",                        "group": "Sports"},
    11: {"label": "Russia-Germany relations",              "group": "IR & Bilateral Relations"},
    12: {"label": "Agricultural industry",                 "group": "Domestic Politics"},
    13: {"label": "Putin's European allies",               "group": "IR & Bilateral Relations"},
    14: {"label": "Energy sector",                         "group": "Domestic Politics"},
    15: {"label": "National award ceremonies",             "group": "Culture & Religion"},
    16: {"label": "WWII commemoration",                    "group": "Military & Security"},
    17: {"label": "Eurasian security cooperation",         "group": "Military & Security"},
    18: {"label": "Military and defense",                  "group": "Military & Security"},
    19: {"label": "Crimea affairs",                        "group": "Domestic Politics"},
    20: {"label": "Russian army",                          "group": "Military & Security"},
    21: {"label": "Religion",                              "group": "Culture & Religion"},
    22: {"label": "Russia-Latin America relations",        "group": "IR & Bilateral Relations"},
    23: {"label": "Domestic emergency response",           "group": "Domestic Politics"},
    24: {"label": "Russia's transportation system",        "group": "Domestic Politics"},
    25: {"label": "Technological innovation",              "group": "Science & Education"},
    26: {"label": "Armed Forces",                          "group": "Military & Security"},
    27: {"label": "Middle-Eastern partnership",            "group": "IR & Bilateral Relations"},
    28: {"label": "Security services",                     "group": "Military & Security"},
    29: {"label": "Nuclear industry",                      "group": "Military & Security"},
    30: {"label": "New Year's speeches",                   "group": "Culture & Religion"},
    31: {"label": "Russia-European relations",             "group": "IR & Bilateral Relations"},
    32: {"label": "Scientific developments",               "group": "Science & Education"},
    33: {"label": "Non-Western allies",                    "group": "IR & Bilateral Relations"},
    34: {"label": "Russia's culture",                      "group": "Culture & Religion"},
    35: {"label": "Russia-Africa relations",               "group": "IR & Bilateral Relations"},
    36: {"label": "Russia's navy",                         "group": "Military & Security"},
    37: {"label": "Civil society and human rights",        "group": "Domestic Politics"},
    38: {"label": "Russia-India relations",                "group": "IR & Bilateral Relations"},
    39: {"label": "Russia-Georgia relations",              "group": "IR & Bilateral Relations"},
    40: {"label": "Russia-ASEAN relations",                "group": "IR & Bilateral Relations"},
    41: {"label": "Russia-Israel-Palestine relations",     "group": "IR & Bilateral Relations"},
    42: {"label": "Space industry",                        "group": "Domestic Politics"},
    43: {"label": "Domestic courts",                       "group": "Executive, Legislature, Judicial"},
    44: {"label": "World Cup",                             "group": "Sports"},
    45: {"label": "Construction industry",                 "group": "Domestic Politics"},
    46: {"label": "Environmental protection",              "group": "Domestic Politics"},
    47: {"label": "Demographic policy",                    "group": "Domestic Politics"},
    48: {"label": "Russia-Scandinavia relations",          "group": "IR & Bilateral Relations"},
    49: {"label": "Interethnic relations",                 "group": "Domestic Politics"},
    50: {"label": "Russia-Mongolia relations",             "group": "IR & Bilateral Relations"},
    51: {"label": "Technology",                            "group": "Science & Education"},
    52: {"label": "Russia-Middle Eastern relations",       "group": "IR & Bilateral Relations"},
    53: {"label": "Russia-Eurasian cooperation",           "group": "IR & Bilateral Relations"},
    54: {"label": "Aviation industry",                     "group": "Military & Security"},
    55: {"label": "Russia-Greece-Cyprus relations",        "group": "IR & Bilateral Relations"},
    56: {"label": "Domestic unions",                       "group": "Domestic Politics"},
    57: {"label": "Russia-Afghanistan relations",          "group": "IR & Bilateral Relations"},
    58: {"label": "Domestic volunteerism",                 "group": "Domestic Politics"},
    59: {"label": "Global economy",                        "group": "IR & Bilateral Relations"},
    60: {"label": "Anti-corruption policy",                "group": "Domestic Politics"},
    61: {"label": "Russian Far East",                      "group": "Domestic Politics"},
    62: {"label": "Elections",                             "group": "Domestic Politics"},
    63: {"label": "Senior citizens",                       "group": "Domestic Politics"},
    64: {"label": "Olympics",                              "group": "Sports"},
    65: {"label": "Russia-Bulgaria relations",             "group": "IR & Bilateral Relations"},
    66: {"label": "Narcotics control",                     "group": "Military & Security"},
    67: {"label": "Taxation",                              "group": "Domestic Politics"},
    68: {"label": "Border control",                        "group": "Military & Security"},
    69: {"label": "Financial monitoring",                  "group": "Domestic Politics"},
    70: {"label": "Russia's geographical society",         "group": "Domestic Politics"},
    71: {"label": "Russia-Brazil relations",               "group": "IR & Bilateral Relations"},
    72: {"label": "Media",                                 "group": "Domestic Politics"},
    73: {"label": "Investments",                           "group": "Domestic Politics"},
    74: {"label": "Russia-Indonesia relations",            "group": "IR & Bilateral Relations"},
    75: {"label": "Russia-Cuba relations",                 "group": "IR & Bilateral Relations"},
    76: {"label": "Arctic exploration",                    "group": "Military & Security"},
    77: {"label": "Caspian region",                        "group": "Military & Security"},
    78: {"label": "Customs service",                       "group": "Military & Security"},
    79: {"label": "Russia-Poland relations",               "group": "IR & Bilateral Relations"},
    80: {"label": "Nazi atrocities",                       "group": "Military & Security"},
    81: {"label": "Students",                              "group": "Science & Education"},
    82: {"label": "Social policy",                         "group": "Domestic Politics"},
    83: {"label": "Martial arts",                          "group": "Sports"},
    84: {"label": "Family affairs",                        "group": "Domestic Politics"},
    85: {"label": "Women's recognition",                   "group": "Domestic Politics"},
    86: {"label": "Land policy",                           "group": "Domestic Politics"},
    87: {"label": "Russia-UN relations",                   "group": "IR & Bilateral Relations"},
    88: {"label": "Russia-Canada relations",               "group": "IR & Bilateral Relations"},
}

# MID RUSSIAN
MID_RU_TOPIC_DICT = {
    0:  {"label": "Ukrainian affairs",              "group": "Post-Soviet Relations"},
    1:  {"label": "Middle Eastern affairs",         "group": "IR & Bilateral Relations"},
    2:  {"label": "Asian affairs",                  "group": "IR & Bilateral Relations"},
    3:  {"label": "Compatriots affairs",            "group": "Post-Soviet Relations"},
    4:  {"label": "African affairs",                "group": "IR & Bilateral Relations"},
    5:  {"label": "Central Asian affairs",          "group": "Post-Soviet Relations"},
    6:  {"label": "Latin American allies",          "group": "IR & Bilateral Relations"},
    7:  {"label": "Iranian nuclear affairs",        "group": "IR & Bilateral Relations"},
    8:  {"label": "European security",              "group": "IR & Bilateral Relations"},
    9:  {"label": "Religion",                       "group": "Internal affairs"},
    10: {"label": "Black Sea cooperation",          "group": "IR & Bilateral Relations"},
    11: {"label": "South Caucasus affairs",         "group": "Post-Soviet Relations"},
    12: {"label": "Arctic affairs",                 "group": "IR & Bilateral Relations"},
    13: {"label": "Eurasian intergovernemntal cooperation", "group": "Post-Soviet Relations"},
    14: {"label": "Developing nations cooperation", "group": "IR & Bilateral Relations"},
    15: {"label": "Cyprus-Greece affairs",          "group": "IR & Bilateral Relations"},
    16: {"label": "Afghanistan relations",          "group": "IR & Bilateral Relations"},
    17: {"label": "WWII commemoration",             "group": "Internal affairs"},
    18: {"label": "Korean affairs",                 "group": "IR & Bilateral Relations"},
    19: {"label": "Nazism",                         "group": "IR & Bilateral Relations"},
    20: {"label": "Russia-Germany relations",       "group": "IR & Bilateral Relations"},
    21: {"label": "Domestic sport",                 "group": "Internal affairs"},
    22: {"label": "ASEAN relations",                "group": "IR & Bilateral Relations"},
    23: {"label": "Regional policy",                "group": "Internal affairs"},
    24: {"label": "MGIMO",                          "group": "Internal affairs"},
    25: {"label": "UNESCO",                         "group": "IR & Bilateral Relations"},
    26: {"label": "Lavrov's interviews",            "group": "IR & Bilateral Relations"},
    27: {"label": "Russia-Italy relations",         "group": "IR & Bilateral Relations"},
    28: {"label": "Caspian region",                 "group": "Post-Soviet Relations"},
    29: {"label": "Anti-terrorist cooperation",     "group": "IR & Bilateral Relations"},
    30: {"label": "Anti-narcotics trafficking",     "group": "IR & Bilateral Relations"},
    31: {"label": "Humanitarian cooperation",       "group": "IR & Bilateral Relations"},
}

In [ ]:
import pandas as pd
import numpy as np
import json, ast, os, re
from pathlib import Path

CORPUS = "kremlin_en"  # one of: "kremlin_en", "kremlin_ru", "mid_en", "mid_ru"

MASTER_CSV = "" # use the master csv file path
SPEECH_TOPK_CSV = "" # use the top k topics csv file path

# If you don't want image columns, set IMAGE_TOPK_CSV = None
IMAGE_TOPK_CSV = "" # use the image top k topics csv file path

OUT_CSV = ""


# -----------------------
TOPIC_DICT_BY_CORPUS = {
    "kremlin_en": KREMLIN_EN_TOPIC_DICT,
    "kremlin_ru": KREMLIN_RU_TOPIC_DICT,
    "mid_en": MID_EN_TOPIC_DICT,
    "mid_ru": MID_RU_TOPIC_DICT,
}
TOPIC_DICT = TOPIC_DICT_BY_CORPUS[CORPUS]

LABEL_MAP = {int(k): v.get("label", None) for k, v in TOPIC_DICT.items()}
GROUP_MAP = {int(k): v.get("group", None) for k, v in TOPIC_DICT.items()}


def _std_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    return df

def _find_id_col(df: pd.DataFrame) -> str:
    for c in ["id", "ID", "Id"]:
        if c in df.columns:
            return c
    raise ValueError(f"Could not find an id column in: {list(df.columns)}")

def _parse_list_cell(x):
    """Parse stored_image_filepaths cell into a python list safely."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return x
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return []
    # try JSON first, then python literal
    try:
        return json.loads(s)
    except Exception:
        try:
            return ast.literal_eval(s)
        except Exception:
            # fallback: treat it as a single path string
            return [s]

def _norm_slashes(p: str) -> str:
    return str(p).replace("\\", "/")

def _normalize_rel_path(p: str, sid: str) -> str:
    """
    Normalize any stored path into a stable key like: "<ID>/<filename>".
    Handles:
      - "1234/img.jpg"
      - "/content/.../1234/img.jpg"
      - "img.jpg"  (will become "1234/img.jpg")
    """
    sid = str(sid)
    s = _norm_slashes(p).lstrip("/").strip()

    # If the string contains "<sid>/", keep from there
    idx = s.find(f"{sid}/")
    if idx != -1:
        return s[idx:]

    # If it already looks like "<sid>/file"
    if re.fullmatch(rf"{re.escape(sid)}/[^/]+", s):
        return s

    # If it's just a filename, prefix with sid
    if "/" not in s:
        return f"{sid}/{s}"

    # Otherwise, fallback: basename with sid
    return f"{sid}/{os.path.basename(s)}"

def _image_row_to_rel(image_path: str, sid: str) -> str:
    """Convert image_topk csv Image_path to rel '<ID>/<filename>'."""
    return _normalize_rel_path(image_path, sid)

# LOAD MASTER + SPEECH TOP-1
master = _std_cols(pd.read_csv(MASTER_CSV))
master_id_col = _find_id_col(master)
master[master_id_col] = master[master_id_col].astype(str).str.strip()

speech = _std_cols(pd.read_csv(SPEECH_TOPK_CSV))
speech_id_col = _find_id_col(speech)
speech[speech_id_col] = speech[speech_id_col].astype(str).str.strip()

# expected columns in speech_topk_topics.csv
need_any = {"top_topic", "top_prob"}
missing = [c for c in need_any if c not in speech.columns]
if missing:
    raise ValueError(f"speech_topk_topics.csv is missing columns {missing}. Found: {list(speech.columns)}")

# Clean numeric
speech["top_topic"] = pd.to_numeric(speech["top_topic"], errors="coerce")
speech["top_prob"]  = pd.to_numeric(speech["top_prob"],  errors="coerce")

# Keep best row per ID (just in case duplicates exist)
speech = (
    speech.sort_values(["top_prob"], ascending=False)
          .drop_duplicates(subset=[speech_id_col], keep="first")
)

master = master.merge(
    speech[[speech_id_col, "top_topic", "top_prob"]],
    how="left",
    left_on=master_id_col,
    right_on=speech_id_col
)

# BUILD CURATED TEXT COLUMNS
# Treat topic = -1 or NaN as missing
cur_topic = master["top_topic"].where(master["top_topic"].notna(), np.nan)
cur_topic = cur_topic.where(cur_topic >= 0, np.nan)

# store as pandas nullable Int
master["curated_topic_id"] = cur_topic.round(0).astype("Int64")

master["curated_topic_probability"] = master["top_prob"].where(master["curated_topic_id"].notna(), np.nan)

master["curated_text_topic_label"] = master["curated_topic_id"].astype("float").map(
    lambda x: LABEL_MAP.get(int(x)) if pd.notna(x) else np.nan
)

master["curated_text_topic_group"] = master["curated_topic_id"].astype("float").map(
    lambda x: GROUP_MAP.get(int(x)) if pd.notna(x) else np.nan
)

# IMAGE TOP-1 PER IMAGE
if IMAGE_TOPK_CSV is not None:
    img = _std_cols(pd.read_csv(IMAGE_TOPK_CSV))
    img_id_col = _find_id_col(img)

    # expected cols (based on your screenshot)
    # ID, Image_path, topic, score ...
    # some files may use "image_path" instead of "Image_path"
    if "Image_path" in img.columns:
        path_col = "Image_path"
    elif "image_path" in img.columns:
        path_col = "image_path"
    else:
        raise ValueError(f"Image csv missing Image_path/image_path. Found: {list(img.columns)}")

    if "topic" not in img.columns:
        raise ValueError(f"Image csv missing 'topic'. Found: {list(img.columns)}")

    # score might be called "score" (as in screenshot)
    if "score" not in img.columns:
        raise ValueError(f"Image csv missing 'score'. Found: {list(img.columns)}")

    img[img_id_col] = img[img_id_col].astype(str).str.strip()
    img["topic"] = pd.to_numeric(img["topic"], errors="coerce")
    img["score"] = pd.to_numeric(img["score"], errors="coerce")

    # Build rel_path key: "<ID>/<filename>"
    img["rel_path"] = img.apply(lambda r: _image_row_to_rel(r[path_col], r[img_id_col]), axis=1)

    # Pick TOP-1 topic per (ID, rel_path) by max score
    img_best = (
        img.sort_values(["score"], ascending=False)
           .drop_duplicates(subset=[img_id_col, "rel_path"], keep="first")
           [[img_id_col, "rel_path", "topic", "score"]]
           .copy()
    )

    # Create lookup: rel_path -> (topic, score)
    # (rel_path already includes ID, so it's unique across corpus)
    img_lookup = {
        row["rel_path"]: (int(row["topic"]) if pd.notna(row["topic"]) else None,
                          float(row["score"]) if pd.notna(row["score"]) else None)
        for _, row in img_best.iterrows()
    }

    # Ensure master has stored_image_filepaths
    if "stored_image_filepaths" not in master.columns:
        raise ValueError("MASTER_CSV does not have 'stored_image_filepaths' column.")

    def build_image_arrays(row):
        sid = str(row[master_id_col]).strip()
        paths = _parse_list_cell(row.get("stored_image_filepaths", None))

        topic_ids, labels, groups, probs = [], [], [], []
        for p in paths:
            rel = _normalize_rel_path(p, sid)
            t, sc = img_lookup.get(rel, (None, None))

            # Map to label/group if topic exists
            if t is None:
                topic_ids.append(None)
                labels.append(None)
                groups.append(None)
                probs.append(None)
            else:
                topic_ids.append(int(t))
                labels.append(LABEL_MAP.get(int(t)))
                groups.append(GROUP_MAP.get(int(t)))
                probs.append(float(sc) if sc is not None else None)

        # store as JSON arrays (safe + consistent)
        return (
            json.dumps(topic_ids, ensure_ascii=False),
            json.dumps(labels,    ensure_ascii=False),
            json.dumps(groups,    ensure_ascii=False),
            json.dumps(probs,     ensure_ascii=False),
        )

    out_cols = master.apply(build_image_arrays, axis=1, result_type="expand")
    out_cols.columns = [
        "curated_image_topic_ids",
        "curated_image_topic_labels",
        "curated_image_group_names",
        "curated_image_topic_probabilities",
    ]
    master = pd.concat([master, out_cols], axis=1)
else:
    # If no image file, still create empty columns
    master["curated_image_topic_ids"] = np.nan
    master["curated_image_topic_labels"] = np.nan
    master["curated_image_group_names"] = np.nan
    master["curated_image_topic_probabilities"] = np.nan

# Drop helper merge key if it was duplicated
if speech_id_col in master.columns and speech_id_col != master_id_col:
    master = master.drop(columns=[speech_id_col])

# we willdrop these intermediate columns
for c in ["top_topic", "top_prob"]:
    if c in master.columns:
        master.drop(columns=[c], inplace=True)

master.to_csv(OUT_CSV, index=False)
print("Wrote:", OUT_CSV)
print("Columns added:",
      ["curated_topic_id","curated_text_topic_label","curated_text_topic_group","curated_topic_probability",
       "curated_image_topic_ids","curated_image_topic_labels","curated_image_group_names","curated_image_topic_probabilities"])

## Export Long-Format CSVs (Text + Image)

1) **Text long format**: 1 row per (speech × topic) with probability
   → if N speeches and K topics ⇒ **N×K rows**
2) **Image long format**: 1 row per (image × topic) with probability
   → if M images and K topics ⇒ **M×K rows**


In [ ]:
# Long Speech (N*K)

!pip -q install -U sentence-transformers transformers tqdm

import os, re
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

SPEECH_CSV = "/content/drive/MyDrive/Russian Speech Dataset Project/New Files/Mid/Russian/CSV Files/mid_russian_english_translated.csv"
FORCED_CSV = "/content/drive/MyDrive/Russian Speech Dataset Project/New Files/Mid/Russian/Bert Model Outputs/Bert Model Outputs/speech_topk_topics_forced_no_minus1.csv"

OUT_DIR = "/content/drive/MyDrive/Russian Speech Dataset Project/New Files/Mid/Russian/Bert Model Outputs"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_TEXTS_CSV    = os.path.join(OUT_DIR, "mid_ru_en_speech_texts.csv")
OUT_LONG_PROBS   = os.path.join(OUT_DIR, "mid_ru_topic_probs_long.csv")
OUT_MISSING_TEXT = os.path.join(OUT_DIR, "ids_with_missing_doc_text.csv")
OUT_IMPOSSIBLE   = os.path.join(OUT_DIR, "ids_where_forced_top_prob_too_small_math_impossible.csv")

INCLUDE_TEXT_IN_LONG = False
OUT_LONG_WITH_TEXT   = os.path.join(OUT_DIR, "mid_ru_topic_probs_long_WITH_TEXT.csv")

TEXT_MODEL = "sentence-transformers/all-mpnet-base-v2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Chunking: full speeches (no truncation)
CHUNK_BODY = 448
CHUNK_STRIDE = 128
MAX_CHUNKS_PER_DOC = 32
BATCH_TEXT = 32

# Adaptive flattening to keep assigned topic as highest probability (when mathematically possible)
MAX_TEMP_ITERS = 12
TEMP_MULT = 1.7
EPS_TOP = 1e-9

# TOPIC METADATA (0..31) MID RU, like this we will give the inputs for the remaining corpus
TOPIC_META = {
  0:  {"topic_name": "Ukrainian affairs",                  "group_name": "Post-Soviet Relations"},
  1:  {"topic_name": "Middle Eastern affairs",             "group_name": "IR & Bilateral Relations"},
  2:  {"topic_name": "Asian affairs",                      "group_name": "IR & Bilateral Relations"},
  3:  {"topic_name": "Compatriots affairs",                "group_name": "Post-Soviet Relations"},
  4:  {"topic_name": "African affairs",                    "group_name": "IR & Bilateral Relations"},
  5:  {"topic_name": "Central Asian affairs",              "group_name": "Post-Soviet Relations"},
  6:  {"topic_name": "Latin American allies",              "group_name": "IR & Bilateral Relations"},
  7:  {"topic_name": "Iranian nuclear affairs",            "group_name": "IR & Bilateral Relations"},
  8:  {"topic_name": "European security",                  "group_name": "IR & Bilateral Relations"},
  9:  {"topic_name": "Religion",                           "group_name": "Internal affairs"},
  10: {"topic_name": "Black Sea cooperation",              "group_name": "IR & Bilateral Relations"},
  11: {"topic_name": "South Caucasus affairs",             "group_name": "Post-Soviet Relations"},
  12: {"topic_name": "Arctic affairs",                     "group_name": "IR & Bilateral Relations"},
  13: {"topic_name": "Eurasian intergovernemntal cooperation","group_name": "Post-Soviet Relations"},
  14: {"topic_name": "Developing nations cooperation",     "group_name": "IR & Bilateral Relations"},
  15: {"topic_name": "Cyprus-Greece affairs",              "group_name": "IR & Bilateral Relations"},
  16: {"topic_name": "Afghanistan relations",              "group_name": "IR & Bilateral Relations"},
  17: {"topic_name": "WWII commemoration",                 "group_name": "Internal affairs"},
  18: {"topic_name": "Korean affairs",                     "group_name": "IR & Bilateral Relations"},
  19: {"topic_name": "Nazism",                             "group_name": "IR & Bilateral Relations"},
  20: {"topic_name": "Russia-Germany relations",           "group_name": "IR & Bilateral Relations"},
  21: {"topic_name": "Domestic sport",                     "group_name": "Internal affairs"},
  22: {"topic_name": "ASEAN relations",                    "group_name": "IR & Bilateral Relations"},
  23: {"topic_name": "Regional policy",                    "group_name": "Internal affairs"},
  24: {"topic_name": "MGIMO",                              "group_name": "Internal affairs"},
  25: {"topic_name": "UNESCO",                             "group_name": "IR & Bilateral Relations"},
  26: {"topic_name": "Lavrov's interviews",                "group_name": "IR & Bilateral Relations"},
  27: {"topic_name": "Russia-Italy relations",             "group_name": "IR & Bilateral Relations"},
  28: {"topic_name": "Caspian region",                     "group_name": "Post-Soviet Relations"},
  29: {"topic_name": "Anti-terrorist cooperation",         "group_name": "IR & Bilateral Relations"},
  30: {"topic_name": "Anti-narcotics trafficking",         "group_name": "IR & Bilateral Relations"},
  31: {"topic_name": "Humanitarian cooperation",           "group_name": "IR & Bilateral Relations"},
}

topic_ids = list(range(0, 32))
K = len(topic_ids)
assert K == 32, f"Expected 32 topics, got {K}"

tid_to_index = {tid: tid for tid in topic_ids}
tid_to_name  = {tid: TOPIC_META[tid]["topic_name"] for tid in topic_ids}
tid_to_group = {tid: TOPIC_META[tid]["group_name"] for tid in topic_ids}

def norm_id(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s:
        return None
    if s.lower() in {"id", "nan", "none"}:
        return None
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    m = re.search(r"(\d+)(?!.*\d)", s)
    if m:
        s = m.group(1)
    s = s.lstrip("0") or "0"
    return s

def clean_text(t: str) -> str:
    t = str(t).replace("\xa0", " ")
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def softmax_rows(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    x = x - np.max(x, axis=1, keepdims=True)
    np.exp(x, out=x)
    s = np.sum(x, axis=1, keepdims=True)
    s[s == 0.0] = 1.0
    x /= s
    return x

def normalize_rows(x: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(x, axis=1, keepdims=True)
    n[n == 0.0] = 1.0
    return x / n

def chunk_text(tokenizer, text: str):
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    chunks = []
    for start in range(0, len(ids), CHUNK_STRIDE):
        window = ids[start:start + CHUNK_BODY]
        if not window:
            break
        ch = tokenizer.decode(window, skip_special_tokens=True).strip()
        if ch:
            chunks.append(ch)
        if len(chunks) >= MAX_CHUNKS_PER_DOC:
            break
    if not chunks:
        chunks = [text[:2000]]
    return chunks

speeches = pd.read_csv(SPEECH_CSV, encoding="utf-8")
forced  = pd.read_csv(FORCED_CSV, encoding="utf-8")

# Required speech columns (we use the English-translated fields)
need_cols = {"ID", "title_en_argos", "full_text_en_argos"}
missing = need_cols - set(speeches.columns)
if missing:
    raise ValueError(f"MID RU speech CSV missing columns: {missing}")

if "ID" not in forced.columns:
    raise ValueError("Forced CSV must contain column 'ID'.")

topic_col = "top_topic_forced" if "top_topic_forced" in forced.columns else "top_topic"
prob_col  = "top_prob_forced"  if "top_prob_forced"  in forced.columns else "top_prob"
if topic_col not in forced.columns or prob_col not in forced.columns:
    raise ValueError(f"Forced CSV must contain {topic_col} and {prob_col}.")

forced["ID"] = forced["ID"].astype(str).str.strip()
forced = forced[forced["ID"] != "ID"].copy()  # drop header-as-row bug if present

forced[topic_col] = pd.to_numeric(forced[topic_col], errors="coerce")
forced[prob_col]  = pd.to_numeric(forced[prob_col],  errors="coerce")

neg_before = int((forced[topic_col] == -1).sum())
if neg_before > 0:
    forced.loc[forced[topic_col] == -1, topic_col] = 0
    print(f"Fixed {neg_before} rows: {topic_col} -1 -> 0 (in-place overwrite)")

forced.to_csv(FORCED_CSV, index=False, encoding="utf-8")
print("Forced CSV overwritten in-place:", FORCED_CSV)

# NORMALIZE IDs + MERGE TEXT
speeches["ID_norm"] = speeches["ID"].map(norm_id)
forced["ID_norm"]   = forced["ID"].map(norm_id)

speeches = speeches.dropna(subset=["ID_norm"]).copy()
forced   = forced.dropna(subset=["ID_norm"]).copy()

if speeches["ID_norm"].nunique() != len(speeches):
    raise ValueError("Speech CSV has duplicate IDs after normalization (unexpected).")
if forced["ID_norm"].nunique() != len(forced):
    raise ValueError("Forced CSV has duplicate IDs after normalization (unexpected).")

df = forced.merge(
    speeches[["ID_norm", "title_en_argos", "full_text_en_argos"]],
    on="ID_norm",
    how="left",
    validate="one_to_one"
)

df["ID"] = df["ID_norm"]

df["title_en_argos"] = df["title_en_argos"].fillna("").astype(str)
df["full_text_en_argos"] = df["full_text_en_argos"].fillna("").astype(str)

missing_mask = df["title_en_argos"].str.strip().eq("") & df["full_text_en_argos"].str.strip().eq("")
missing_ids = df.loc[missing_mask, "ID"].tolist()
print("IDs missing BOTH title_en_argos and full_text_en_argos:", len(missing_ids))

pd.DataFrame({"ID": missing_ids}).to_csv(OUT_MISSING_TEXT, index=False)
print("Saved missing doc_text IDs ->", OUT_MISSING_TEXT)

df["doc_text"] = (df["title_en_argos"].str.strip() + ". " + df["full_text_en_argos"].str.strip()).str.strip(". ").map(clean_text)

pd.DataFrame({
    "ID": df["ID"],
    "title_en_argos": df["title_en_argos"],
    "full_text_en_argos": df["full_text_en_argos"],
}).to_csv(OUT_TEXTS_CSV, index=False)
print("Saved speech texts ->", OUT_TEXTS_CSV)

df[topic_col] = pd.to_numeric(df[topic_col], errors="coerce").astype(int)
df[prob_col]  = pd.to_numeric(df[prob_col],  errors="coerce").astype(float)

assigned_tid = df[topic_col].values.astype(int)
forced_p = np.clip(df[prob_col].values.astype(np.float32), 0.0, 1.0)

bad_topics = sorted(set(assigned_tid) - set(topic_ids))
if bad_topics:
    raise ValueError(f"Assigned topics outside 0..{K-1}: {bad_topics}")

# EMBEDDING (FULL SPEECH VIA CHUNKING)
print("Loading embedding model:", TEXT_MODEL)
model = SentenceTransformer(TEXT_MODEL, device=DEVICE)
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL, use_fast=True)

texts = df["doc_text"].tolist()

chunk_lists = []
for t in tqdm(texts, desc="Chunking speeches", total=len(texts)):
    chunk_lists.append(chunk_text(tokenizer, t))

dim = model.get_sentence_embedding_dimension()
N = len(df)
doc_vecs = np.zeros((N, dim), dtype=np.float32)

work = []
for i, chunks in enumerate(chunk_lists):
    for ch in chunks:
        work.append((i, ch))

bucket = [[] for _ in range(N)]

print("Encoding chunks...")
cursor = 0
with tqdm(total=len(work), desc="Encode chunks") as bar:
    while cursor < len(work):
        end = min(cursor + BATCH_TEXT, len(work))
        owners = [work[j][0] for j in range(cursor, end)]
        batch_txt = [work[j][1] for j in range(cursor, end)]
        vecs = model.encode(
            batch_txt,
            batch_size=len(batch_txt),
            device=DEVICE,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=False
        ).astype(np.float32)
        for o, v in zip(owners, vecs):
            bucket[o].append(v)
        cursor = end
        bar.update(len(batch_txt))

for i in range(N):
    if bucket[i]:
        doc_vecs[i] = np.mean(bucket[i], axis=0)
    else:
        doc_vecs[i] = model.encode([texts[i][:2000]], device=DEVICE, show_progress_bar=False, convert_to_numpy=True)[0].astype(np.float32)

doc_vecs = normalize_rows(doc_vecs)

# TOPIC CENTROIDS (FROM FINAL ASSIGNMENTS)
centroids = np.zeros((K, dim), dtype=np.float32)
for tid in topic_ids:
    idxs = np.where(assigned_tid == tid)[0]
    if len(idxs) == 0:
        raise ValueError(f"Topic {tid} has 0 assigned speeches (you said this should never happen).")
    centroids[tid_to_index[tid]] = np.mean(doc_vecs[idxs], axis=0)

centroids = normalize_rows(centroids)

# COSINE SCORES
scores = np.dot(doc_vecs, centroids.T).astype(np.float32)  # N x K

# BUILD P (N x K) WITH FIXED ASSIGNED PROB
P = np.zeros((N, K), dtype=np.float32)

p_min = 1.0 / K
feasible = forced_p > (p_min + 1e-12)

impossible_ids = df.loc[~feasible, "ID"].tolist()
pd.DataFrame({"ID": impossible_ids, "top_prob_forced": forced_p[~feasible]}).to_csv(OUT_IMPOSSIBLE, index=False)
print(f"Math-impossible to keep assigned strictly top (top_prob_forced <= 1/{K}): {len(impossible_ids)}")
print("Saved ->", OUT_IMPOSSIBLE)

tau = np.ones(N, dtype=np.float32)

def compute_P_for_rows(idxs):
    logits = scores[idxs] / tau[idxs][:, None]
    q = softmax_rows(logits)

    j_star = np.array([tid_to_index[int(t)] for t in assigned_tid[idxs]], dtype=int)
    q[np.arange(len(idxs)), j_star] = 0.0
    s = q.sum(axis=1, keepdims=True)

    q_norm = np.where(s > 1e-12, q / s, 0.0)
    if np.any(s <= 1e-12):
        bad = np.where((s[:,0] <= 1e-12))[0]
        for b in bad:
            q_norm[b, :] = 1.0 / (K - 1)
            q_norm[b, j_star[b]] = 0.0

    remain = (1.0 - forced_p[idxs]).astype(np.float32)[:, None]
    P_sub = remain * q_norm
    P_sub[np.arange(len(idxs)), j_star] = forced_p[idxs]
    return P_sub, j_star

idx_impossible = np.where(~feasible)[0]
if len(idx_impossible) > 0:
    for i in idx_impossible:
        j = tid_to_index[int(assigned_tid[i])]
        P[i, :] = (1.0 - forced_p[i]) / (K - 1)
        P[i, j] = forced_p[i]

active = feasible.copy()
for _ in range(MAX_TEMP_ITERS):
    idxs = np.where(active)[0]
    if len(idxs) == 0:
        break

    P_sub, j_star = compute_P_for_rows(idxs)

    tmp = P_sub.copy()
    tmp[np.arange(len(idxs)), j_star] = -1.0
    max_other = tmp.max(axis=1)
    p_star = forced_p[idxs]

    viol = max_other >= (p_star - EPS_TOP)

    ok_mask = ~viol
    if np.any(ok_mask):
        ok_idxs = idxs[ok_mask]
        P[ok_idxs] = P_sub[ok_mask]
        active[ok_idxs] = False

    if np.any(viol):
        viol_idxs = idxs[viol]
        tau[viol_idxs] *= TEMP_MULT

idx_left = np.where(active)[0]
if len(idx_left) > 0:
    tau[idx_left] = 1e6
    P_left, _ = compute_P_for_rows(idx_left)
    P[idx_left] = P_left
    active[idx_left] = False

# VALIDATIONS
j_all = np.array([tid_to_index[int(t)] for t in assigned_tid], dtype=int)
assigned_from_P = P[np.arange(N), j_all]

print("Max abs error (assigned prob vs forced):", float(np.max(np.abs(assigned_from_P - forced_p))))
print("Max row-sum error:", float(np.max(np.abs(P.sum(axis=1) - 1.0))))

tmp = P.copy()
tmp[np.arange(N), j_all] = -1.0
max_other_all = tmp.max(axis=1)
viol_count = int((max_other_all >= (forced_p - EPS_TOP)).sum())
print("Rows where some other topic >= assigned prob:", viol_count)

# BUILD LONG OUTPUT (N*K)
print(f"Building long output (N*K = {N}*{K} = {N*K}) ...")
rows = []
ids = df["ID"].tolist()

for i, sid in enumerate(tqdm(ids, total=N)):
    for tid in topic_ids:
        rows.append({
            "ID": sid,
            "Topic ID": tid,
            "Topic Name": tid_to_name[tid],
            "Topic Group Name": tid_to_group[tid],
            "Probability score": float(P[i, tid_to_index[tid]]),
        })

long_df = pd.DataFrame(rows)
long_df.to_csv(OUT_LONG_PROBS, index=False, encoding="utf-8")
print("Saved long probs ->", OUT_LONG_PROBS)

if INCLUDE_TEXT_IN_LONG:
    print("WARNING: Writing long WITH_TEXT can be very large (text repeated K times).")
    texts_df = pd.read_csv(OUT_TEXTS_CSV, encoding="utf-8")[["ID", "title_en_argos", "full_text_en_argos"]]
    long_with_text = long_df.merge(texts_df, on="ID", how="left", validate="many_to_one")
    long_with_text = long_with_text[["ID", "title_en_argos", "full_text_en_argos", "Topic ID", "Topic Name", "Topic Group Name", "Probability score"]]
    long_with_text.to_csv(OUT_LONG_WITH_TEXT, index=False, encoding="utf-8")
    print("Saved long WITH_TEXT ->", OUT_LONG_WITH_TEXT)

print("DONE.")

# Lomg format image probability files

In [ ]:
!pip -q install gspread google-auth open_clip_torch pillow tqdm

from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")

import os, re, json, ast, csv
from tqdm import tqdm

import torch
import torch.nn.functional as F
from PIL import Image

import gspread
from google.auth import default
import open_clip


SHEET_URL = "https://docs.google.com/spreadsheets/d/1vAPXKWZ26O-Xs4GRN8WUMM452eoquGXRdM_An2nTU2Q/edit?gid=1134303671#gid=1134303671"
WORKSHEET_NAME = None

IMAGE_BASE_DIR = "/content/drive/MyDrive/Russian Speech Dataset Project/New Files/Mid/Russian/Scraped Images"
IMAGE_LIST_COL = "stored_image_filepaths"  # column holding ["ID/file.jpg", ...]

OUTPUT_CSV_PATH = "/content/drive/MyDrive/Russian Speech Dataset Project/New Files/Final CSV Files/processed/mid_russian_image_topic_probs.csv"

MODEL_NAME = "ViT-H-14"
PRETRAINED = "laion2b_s32b_b79k"

PROMPTS = [
    "{topic}",
    "a photo about {topic}",
    "a photo of {topic}",
    "an image about {topic}",
    "a picture of {topic}",
]

BATCH_SIZE = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# MID (Russian) topic dictionaries, we can do similarly for any of the remaining corpus.
TOPIC_ID_TO_LABEL = {
    0: "Ukrainian affairs",
    1: "Middle Eastern affairs",
    2: "Asian affairs",
    3: "Compatriots affairs",
    4: "African affairs",
    5: "Central Asian affairs",
    6: "Latin American allies",
    7: "Iranian nuclear affairs",
    8: "European security",
    9: "Religion",
    10: "Black Sea cooperation",
    11: "South Caucasus affairs",
    12: "Arctic affairs",
    13: "Eurasian intergovernemntal cooperation",
    14: "Developing nations cooperation",
    15: "Cyprus-Greece affairs",
    16: "Afghanistan relations",
    17: "WWII commemoration",
    18: "Korean affairs",
    19: "Nazism",
    20: "Russia-Germany relations",
    21: "Domestic sport",
    22: "ASEAN relations",
    23: "Regional policy",
    24: "MGIMO",
    25: "UNESCO",
    26: "Lavrov's interviews",
    27: "Russia-Italy relations",
    28: "Caspian region",
    29: "Anti-terrorist cooperation",
    30: "Anti-narcotics trafficking",
    31: "Humanitarian cooperation",
}

TOPIC_ID_TO_GROUP = {
    0: "Post-Soviet Relations",
    1: "IR & Bilateral Relations",
    2: "IR & Bilateral Relations",
    3: "Post-Soviet Relations",
    4: "IR & Bilateral Relations",
    5: "Post-Soviet Relations",
    6: "IR & Bilateral Relations",
    7: "IR & Bilateral Relations",
    8: "IR & Bilateral Relations",
    9: "Internal affairs",
    10: "IR & Bilateral Relations",
    11: "Post-Soviet Relations",
    12: "IR & Bilateral Relations",
    13: "Post-Soviet Relations",
    14: "IR & Bilateral Relations",
    15: "IR & Bilateral Relations",
    16: "IR & Bilateral Relations",
    17: "Internal affairs",
    18: "IR & Bilateral Relations",
    19: "IR & Bilateral Relations",
    20: "IR & Bilateral Relations",
    21: "Internal affairs",
    22: "IR & Bilateral Relations",
    23: "Internal affairs",
    24: "Internal affairs",
    25: "IR & Bilateral Relations",
    26: "IR & Bilateral Relations",
    27: "IR & Bilateral Relations",
    28: "Post-Soviet Relations",
    29: "IR & Bilateral Relations",
    30: "IR & Bilateral Relations",
    31: "IR & Bilateral Relations",
}

missing_groups = [k for k in TOPIC_ID_TO_LABEL.keys() if k not in TOPIC_ID_TO_GROUP]
if missing_groups:
    raise ValueError(f"Missing group names for topic ids: {missing_groups}")


def parse_list_cell(v) -> list:
    if v is None:
        return []
    s = str(v).strip()
    if not s or s.lower() == "nan":
        return []

    if s.startswith("[") and s.endswith("]"):
        try:
            x = json.loads(s)
            if isinstance(x, list):
                return [str(i).strip() for i in x if str(i).strip()]
        except Exception:
            pass
        try:
            x = ast.literal_eval(s)
            if isinstance(x, list):
                return [str(i).strip() for i in x if str(i).strip()]
        except Exception:
            pass

    delim = "|" if "|" in s else ","
    items = [x.strip().strip('"').strip("'") for x in s.split(delim)]
    return [x for x in items if x]


def safe_join(base, rel):
    rel = rel.lstrip("/").replace("\\", "/")
    return os.path.join(base, rel)


def load_image_rgb(path: str) -> Image.Image:
    with Image.open(path) as img:
        return img.convert("RGB")


def get_gid_from_url(url: str):
    m = re.search(r"gid=(\d+)", url)
    return int(m.group(1)) if m else None


def open_sheet_and_tab(sheet_url: str, worksheet_name):
    creds, _ = default()
    gc = gspread.authorize(creds)
    sh = gc.open_by_url(sheet_url)

    if worksheet_name:
        ws = sh.worksheet(worksheet_name)
        return sh, ws

    gid = get_gid_from_url(sheet_url)
    if gid is not None:
        for w in sh.worksheets():
            if w.id == gid:
                return sh, w

    return sh, sh.get_worksheet(0)


sh, ws = open_sheet_and_tab(SHEET_URL, WORKSHEET_NAME)
headers = [h.strip() for h in ws.row_values(1)]
if IMAGE_LIST_COL not in headers:
    raise ValueError(f"Column '{IMAGE_LIST_COL}' not found. Available: {headers}")

img_col_idx = headers.index(IMAGE_LIST_COL) + 1
col_vals = ws.col_values(img_col_idx)[1:]  # skip header

image_refs = []
for cell in col_vals:
    for rel in parse_list_cell(cell):
        if not rel:
            continue
        img_id = rel.split("/", 1)[0].strip()
        image_refs.append((img_id, rel))

print(f"Sheet: {sh.title} / Tab: {ws.title}")
print(f"Total image references from sheet: {len(image_refs)}")


# ALIGNMENT CHECK
exists, missing = 0, 0
missing_samples = []

for img_id, rel in image_refs:
    full = safe_join(IMAGE_BASE_DIR, rel)
    if os.path.exists(full):
        exists += 1
    else:
        missing += 1
        if len(missing_samples) < 20:
            missing_samples.append(full)

print("\n========== ALIGNMENT CHECK ==========")
print(f"Base folder: {IMAGE_BASE_DIR}")
print(f"Exists:  {exists}")
print(f"Missing: {missing}")
if missing_samples:
    print("\nFirst missing samples:")
    for p in missing_samples:
        print("  -", p)

if exists == 0:
    raise RuntimeError("0 images found on Drive for the sheet paths. Base folder/path mismatch.")

image_refs_existing = [(img_id, rel) for (img_id, rel) in image_refs if os.path.exists(safe_join(IMAGE_BASE_DIR, rel))]
print(f"\nProceeding with existing images only: {len(image_refs_existing)}")


# LOAD CLIP + BUILD TOPIC TEXT EMBEDDINGS
topic_ids = sorted(TOPIC_ID_TO_LABEL.keys())
topic_names = [TOPIC_ID_TO_LABEL[k] for k in topic_ids]
topic_groups = [TOPIC_ID_TO_GROUP[k] for k in topic_ids]

print(f"\nTotal topics: {len(topic_ids)}")
print("Loading CLIP model:", MODEL_NAME, PRETRAINED)

model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model = model.to(DEVICE).eval()

@torch.inference_mode()
def build_text_embeddings():
    all_topic_embs = []
    for tname in tqdm(topic_names, desc="Encoding topic texts"):
        prompts = [p.format(topic=tname) for p in PROMPTS]
        toks = tokenizer(prompts).to(DEVICE)
        text_feat = model.encode_text(toks)
        text_feat = F.normalize(text_feat, dim=-1)
        text_feat = text_feat.mean(dim=0, keepdim=True)
        text_feat = F.normalize(text_feat, dim=-1)
        all_topic_embs.append(text_feat)
    return torch.cat(all_topic_embs, dim=0)

text_emb = build_text_embeddings()


# STREAM IMAGES -> PROBS -> WRITE CSV
@torch.inference_mode()
def encode_images_batch(batch_paths):
    imgs = [preprocess(load_image_rgb(p)) for p in batch_paths]
    imgs = torch.stack(imgs, dim=0).to(DEVICE)
    img_feat = model.encode_image(imgs)
    return F.normalize(img_feat, dim=-1)

os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)

bad_images = []
written_images = 0

print("\nWriting output CSV to:", OUTPUT_CSV_PATH)
with open(OUTPUT_CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["ID", "image_path", "topic_id", "topic_name", "topic_group", "probability"])

    total_imgs = len(image_refs_existing)
    for start in tqdm(range(0, total_imgs, BATCH_SIZE), desc="Images"):
        chunk = image_refs_existing[start:start + BATCH_SIZE]
        chunk_fullpaths = [safe_join(IMAGE_BASE_DIR, rel) for _, rel in chunk]

        ok_pairs, ok_paths = [], []
        for (img_id, rel), fullp in zip(chunk, chunk_fullpaths):
            try:
                _ = load_image_rgb(fullp)
                ok_pairs.append((img_id, rel))
                ok_paths.append(fullp)
            except Exception as e:
                bad_images.append((img_id, rel, str(e)))

        if not ok_paths:
            continue

        img_emb = encode_images_batch(ok_paths)
        logit_scale = model.logit_scale.exp()
        logits = logit_scale * (img_emb @ text_emb.T)

        # FIX: detach before numpy (safe even if something tracks grads)
        probs = F.softmax(logits, dim=1).detach().cpu().numpy()

        for (img_id, rel), pvec in zip(ok_pairs, probs):
            written_images += 1
            for tid, tname, gname, pr in zip(topic_ids, topic_names, topic_groups, pvec.tolist()):
                writer.writerow([img_id, rel, tid, tname, gname, pr])

print("\n DONE.")
print(f"Images referenced in sheet: {len(image_refs)}")
print(f"Images used (exist + readable): {written_images}")
print(f"Bad images skipped: {len(bad_images)}")
print(f"Total topics: {len(topic_ids)}")
print(f"Total rows written: {written_images * len(topic_ids)}")
print("CSV saved at:", OUTPUT_CSV_PATH)

if bad_images:
    print("\n========== BAD IMAGES (PRINT ALL) ==========")
    for img_id, rel, err in bad_images:
        print(f"{img_id} | {rel} | {err}")